# DB-SLM: Fine-tuning a Small Language Model on Retail + MIMIC Databases
**Candidate:** Bilal Javed | Adept Tech Solutions — Final Evaluation Task

**Steps:** Install → Check GPU → Build Dataset (33 verified QA pairs) → Load Model (4-bit) → Baseline test → QLoRA → Train → Compare before/after → Save adapter

**Runtime:** `Runtime > Change runtime type > T4 GPU` | **Expected time:** ~15-20 min

## Step 1: Install Dependencies

In [1]:
!pip install -q -U transformers peft trl bitsandbytes datasets accelerate
print('Dependencies installed successfully!')

Dependencies installed successfully!


## Step 2: Check GPU

In [2]:
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    raise RuntimeError('No GPU detected! Please go to Runtime > Change runtime type > T4 GPU')

GPU available: True
Device: Tesla T4
VRAM: 15.6 GB


## Step 3: Build Training Dataset

**33 verified QA pairs** — all answers derived from real SQL queries on actual databases.

Facts confirmed:
- Revenue £1,276,568.60 | Customers 4,372 | Invoices 25,900 (14.8% cancelled)
- Patients 100 (43F/57M) | Admissions 275 | Mortality 5.5% | ICU stays 140 | Avg LOS 3.68 days
- Lab results 107,727 (37.4% abnormal) | Prescriptions 18,087 | Chartevents 668,862

In [3]:
import json, random
random.seed(42)

EXAMPLES = [
  {
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant trained on two real relational databases.\n\nDATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):\nTables:\n  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)\n  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)\n  invoices (25,900 rows): invoice_no PK, customer_id FK, invoice_date, is_cancelled (1=return/refund)\n  invoice_items (54,873 rows): invoice_no FK, stock_code FK, quantity, unit_price, line_total\n  patient_customer_bridge (100 rows): links customer_id to MIMIC subject_id\nRelationships: invoice_items.invoice_no -> invoices | invoice_items.stock_code -> products | invoices.customer_id -> customers\nKey verified facts (from actual SQL on real database):\n  - 4,372 customers across 38 countries\n  - Confirmed order revenue (is_cancelled=0): £1,276,568.60\n  - Cancellation rate: 14.8% (3,836 of 25,900 invoices)\n  - Average confirmed order value: £100.52 per invoice\n  - Top 5 countries by revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194\n  - Date range: 2010-12-01 to 2011-12-09\n\nDATABASE 2 - MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):\nTables:\n  patients (100 rows): subject_id PK, gender M/F, anchor_age, anchor_year_group, dod\n  admissions (275 rows): hadm_id PK, subject_id FK, admission_type, insurance, hospital_expire_flag (1=died)\n  diagnoses_icd (4,506 rows): seq_num=1 is primary diagnosis, icd_version is 9 or 10\n  labevents (107,727 rows): itemid FK, value, valuenum, flag='abnormal' means outside reference range\n  prescriptions (18,087 rows): drug, dose, route, starttime, stoptime\n  icustays (140 rows): stay_id PK, first_careunit, los (length of stay in days)\n  chartevents (668,862 rows): ICU vital signs and physiological measurements\n  + d_labitems, d_icd_diagnoses, d_items reference dictionaries + 18 more tables\nKey verified facts (from actual SQL on real database):\n  - 100 patients: 43 female, 57 male. Mean anchor age: 61.8 years\n  - 31 patients have a recorded date of death (dod is not empty)\n  - In-hospital mortality: 15 deaths / 275 admissions = 5.5%\n  - Average ICU length of stay: 3.68 days (140 stays total)\n  - 107,727 lab results; 37.4% flagged as abnormal (40,275 results)\n  - 18,087 prescriptions | 668,862 chartevents\n  - Admission types: EW EMER, URGENT, ELECTIVE, DIRECT EMER, OBSERVATION ADMIT\n\nCROSS-DATABASE: patient_customer_bridge (100 rows) maps retail customer_id to MIMIC subject_id.\n\nAnswer all questions using these verified facts only."
      },
      {
        "role": "user",
        "content": "Which country generates the most retail revenue?"
      },
      {
        "role": "assistant",
        "content": "United Kingdom with £685,023 confirmed revenue (dominant domestic Domestic_UK segment). Followed by Germany, France, EIRE, Spain."
      }
    ]
  },
  {
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant trained on two real relational databases.\n\nDATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):\nTables:\n  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)\n  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)\n  invoices (25,900 rows): invoice_no PK, customer_id FK, invoice_date, is_cancelled (1=return/refund)\n  invoice_items (54,873 rows): invoice_no FK, stock_code FK, quantity, unit_price, line_total\n  patient_customer_bridge (100 rows): links customer_id to MIMIC subject_id\nRelationships: invoice_items.invoice_no -> invoices | invoice_items.stock_code -> products | invoices.customer_id -> customers\nKey verified facts (from actual SQL on real database):\n  - 4,372 customers across 38 countries\n  - Confirmed order revenue (is_cancelled=0): £1,276,568.60\n  - Cancellation rate: 14.8% (3,836 of 25,900 invoices)\n  - Average confirmed order value: £100.52 per invoice\n  - Top 5 countries by revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194\n  - Date range: 2010-12-01 to 2011-12-09\n\nDATABASE 2 - MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):\nTables:\n  patients (100 rows): subject_id PK, gender M/F, anchor_age, anchor_year_group, dod\n  admissions (275 rows): hadm_id PK, subject_id FK, admission_type, insurance, hospital_expire_flag (1=died)\n  diagnoses_icd (4,506 rows): seq_num=1 is primary diagnosis, icd_version is 9 or 10\n  labevents (107,727 rows): itemid FK, value, valuenum, flag='abnormal' means outside reference range\n  prescriptions (18,087 rows): drug, dose, route, starttime, stoptime\n  icustays (140 rows): stay_id PK, first_careunit, los (length of stay in days)\n  chartevents (668,862 rows): ICU vital signs and physiological measurements\n  + d_labitems, d_icd_diagnoses, d_items reference dictionaries + 18 more tables\nKey verified facts (from actual SQL on real database):\n  - 100 patients: 43 female, 57 male. Mean anchor age: 61.8 years\n  - 31 patients have a recorded date of death (dod is not empty)\n  - In-hospital mortality: 15 deaths / 275 admissions = 5.5%\n  - Average ICU length of stay: 3.68 days (140 stays total)\n  - 107,727 lab results; 37.4% flagged as abnormal (40,275 results)\n  - 18,087 prescriptions | 668,862 chartevents\n  - Admission types: EW EMER, URGENT, ELECTIVE, DIRECT EMER, OBSERVATION ADMIT\n\nCROSS-DATABASE: patient_customer_bridge (100 rows) maps retail customer_id to MIMIC subject_id.\n\nAnswer all questions using these verified facts only."
      },
      {
        "role": "user",
        "content": "What does seq_num=1 mean in diagnoses_icd?"
      },
      {
        "role": "assistant",
        "content": "seq_num=1 is the primary (principal) diagnosis - the main reason for hospital admission. Higher seq_num values are secondary or additional diagnoses. Filter WHERE seq_num=1 for primary diagnosis analysis."
      }
    ]
  },
  {
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant trained on two real relational databases.\n\nDATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):\nTables:\n  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)\n  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)\n  invoices (25,900 rows): invoice_no PK, customer_id FK, invoice_date, is_cancelled (1=return/refund)\n  invoice_items (54,873 rows): invoice_no FK, stock_code FK, quantity, unit_price, line_total\n  patient_customer_bridge (100 rows): links customer_id to MIMIC subject_id\nRelationships: invoice_items.invoice_no -> invoices | invoice_items.stock_code -> products | invoices.customer_id -> customers\nKey verified facts (from actual SQL on real database):\n  - 4,372 customers across 38 countries\n  - Confirmed order revenue (is_cancelled=0): £1,276,568.60\n  - Cancellation rate: 14.8% (3,836 of 25,900 invoices)\n  - Average confirmed order value: £100.52 per invoice\n  - Top 5 countries by revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194\n  - Date range: 2010-12-01 to 2011-12-09\n\nDATABASE 2 - MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):\nTables:\n  patients (100 rows): subject_id PK, gender M/F, anchor_age, anchor_year_group, dod\n  admissions (275 rows): hadm_id PK, subject_id FK, admission_type, insurance, hospital_expire_flag (1=died)\n  diagnoses_icd (4,506 rows): seq_num=1 is primary diagnosis, icd_version is 9 or 10\n  labevents (107,727 rows): itemid FK, value, valuenum, flag='abnormal' means outside reference range\n  prescriptions (18,087 rows): drug, dose, route, starttime, stoptime\n  icustays (140 rows): stay_id PK, first_careunit, los (length of stay in days)\n  chartevents (668,862 rows): ICU vital signs and physiological measurements\n  + d_labitems, d_icd_diagnoses, d_items reference dictionaries + 18 more tables\nKey verified facts (from actual SQL on real database):\n  - 100 patients: 43 female, 57 male. Mean anchor age: 61.8 years\n  - 31 patients have a recorded date of death (dod is not empty)\n  - In-hospital mortality: 15 deaths / 275 admissions = 5.5%\n  - Average ICU length of stay: 3.68 days (140 stays total)\n  - 107,727 lab results; 37.4% flagged as abnormal (40,275 results)\n  - 18,087 prescriptions | 668,862 chartevents\n  - Admission types: EW EMER, URGENT, ELECTIVE, DIRECT EMER, OBSERVATION ADMIT\n\nCROSS-DATABASE: patient_customer_bridge (100 rows) maps retail customer_id to MIMIC subject_id.\n\nAnswer all questions using these verified facts only."
      },
      {
        "role": "user",
        "content": "What are the customer segments in the retail database?"
      },
      {
        "role": "assistant",
        "content": "Three segments: Domestic_UK (UK customers, largest by revenue), EU_Wholesale (European countries outside UK), International (rest of world, smallest segment)."
      }
    ]
  },
  {
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant trained on two real relational databases.\n\nDATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):\nTables:\n  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)\n  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)\n  invoices (25,900 rows): invoice_no PK, customer_id FK, invoice_date, is_cancelled (1=return/refund)\n  invoice_items (54,873 rows): invoice_no FK, stock_code FK, quantity, unit_price, line_total\n  patient_customer_bridge (100 rows): links customer_id to MIMIC subject_id\nRelationships: invoice_items.invoice_no -> invoices | invoice_items.stock_code -> products | invoices.customer_id -> customers\nKey verified facts (from actual SQL on real database):\n  - 4,372 customers across 38 countries\n  - Confirmed order revenue (is_cancelled=0): £1,276,568.60\n  - Cancellation rate: 14.8% (3,836 of 25,900 invoices)\n  - Average confirmed order value: £100.52 per invoice\n  - Top 5 countries by revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194\n  - Date range: 2010-12-01 to 2011-12-09\n\nDATABASE 2 - MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):\nTables:\n  patients (100 rows): subject_id PK, gender M/F, anchor_age, anchor_year_group, dod\n  admissions (275 rows): hadm_id PK, subject_id FK, admission_type, insurance, hospital_expire_flag (1=died)\n  diagnoses_icd (4,506 rows): seq_num=1 is primary diagnosis, icd_version is 9 or 10\n  labevents (107,727 rows): itemid FK, value, valuenum, flag='abnormal' means outside reference range\n  prescriptions (18,087 rows): drug, dose, route, starttime, stoptime\n  icustays (140 rows): stay_id PK, first_careunit, los (length of stay in days)\n  chartevents (668,862 rows): ICU vital signs and physiological measurements\n  + d_labitems, d_icd_diagnoses, d_items reference dictionaries + 18 more tables\nKey verified facts (from actual SQL on real database):\n  - 100 patients: 43 female, 57 male. Mean anchor age: 61.8 years\n  - 31 patients have a recorded date of death (dod is not empty)\n  - In-hospital mortality: 15 deaths / 275 admissions = 5.5%\n  - Average ICU length of stay: 3.68 days (140 stays total)\n  - 107,727 lab results; 37.4% flagged as abnormal (40,275 results)\n  - 18,087 prescriptions | 668,862 chartevents\n  - Admission types: EW EMER, URGENT, ELECTIVE, DIRECT EMER, OBSERVATION ADMIT\n\nCROSS-DATABASE: patient_customer_bridge (100 rows) maps retail customer_id to MIMIC subject_id.\n\nAnswer all questions using these verified facts only."
      },
      {
        "role": "user",
        "content": "How many ICU stays are in MIMIC?"
      },
      {
        "role": "assistant",
        "content": "140 ICU stays across 100 patients. Average ICU length of stay: 3.68 days."
      }
    ]
  },
  {
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant trained on two real relational databases.\n\nDATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):\nTables:\n  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)\n  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)\n  invoices (25,900 rows): invoice_no PK, customer_id FK, invoice_date, is_cancelled (1=return/refund)\n  invoice_items (54,873 rows): invoice_no FK, stock_code FK, quantity, unit_price, line_total\n  patient_customer_bridge (100 rows): links customer_id to MIMIC subject_id\nRelationships: invoice_items.invoice_no -> invoices | invoice_items.stock_code -> products | invoices.customer_id -> customers\nKey verified facts (from actual SQL on real database):\n  - 4,372 customers across 38 countries\n  - Confirmed order revenue (is_cancelled=0): £1,276,568.60\n  - Cancellation rate: 14.8% (3,836 of 25,900 invoices)\n  - Average confirmed order value: £100.52 per invoice\n  - Top 5 countries by revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194\n  - Date range: 2010-12-01 to 2011-12-09\n\nDATABASE 2 - MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):\nTables:\n  patients (100 rows): subject_id PK, gender M/F, anchor_age, anchor_year_group, dod\n  admissions (275 rows): hadm_id PK, subject_id FK, admission_type, insurance, hospital_expire_flag (1=died)\n  diagnoses_icd (4,506 rows): seq_num=1 is primary diagnosis, icd_version is 9 or 10\n  labevents (107,727 rows): itemid FK, value, valuenum, flag='abnormal' means outside reference range\n  prescriptions (18,087 rows): drug, dose, route, starttime, stoptime\n  icustays (140 rows): stay_id PK, first_careunit, los (length of stay in days)\n  chartevents (668,862 rows): ICU vital signs and physiological measurements\n  + d_labitems, d_icd_diagnoses, d_items reference dictionaries + 18 more tables\nKey verified facts (from actual SQL on real database):\n  - 100 patients: 43 female, 57 male. Mean anchor age: 61.8 years\n  - 31 patients have a recorded date of death (dod is not empty)\n  - In-hospital mortality: 15 deaths / 275 admissions = 5.5%\n  - Average ICU length of stay: 3.68 days (140 stays total)\n  - 107,727 lab results; 37.4% flagged as abnormal (40,275 results)\n  - 18,087 prescriptions | 668,862 chartevents\n  - Admission types: EW EMER, URGENT, ELECTIVE, DIRECT EMER, OBSERVATION ADMIT\n\nCROSS-DATABASE: patient_customer_bridge (100 rows) maps retail customer_id to MIMIC subject_id.\n\nAnswer all questions using these verified facts only."
      },
      {
        "role": "user",
        "content": "What are the top countries by retail revenue?"
      },
      {
        "role": "assistant",
        "content": "Top 5 by confirmed revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194."
      }
    ]
  },
  {
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant trained on two real relational databases.\n\nDATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):\nTables:\n  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)\n  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)\n  invoices (25,900 rows): invoice_no PK, customer_id FK, invoice_date, is_cancelled (1=return/refund)\n  invoice_items (54,873 rows): invoice_no FK, stock_code FK, quantity, unit_price, line_total\n  patient_customer_bridge (100 rows): links customer_id to MIMIC subject_id\nRelationships: invoice_items.invoice_no -> invoices | invoice_items.stock_code -> products | invoices.customer_id -> customers\nKey verified facts (from actual SQL on real database):\n  - 4,372 customers across 38 countries\n  - Confirmed order revenue (is_cancelled=0): £1,276,568.60\n  - Cancellation rate: 14.8% (3,836 of 25,900 invoices)\n  - Average confirmed order value: £100.52 per invoice\n  - Top 5 countries by revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194\n  - Date range: 2010-12-01 to 2011-12-09\n\nDATABASE 2 - MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):\nTables:\n  patients (100 rows): subject_id PK, gender M/F, anchor_age, anchor_year_group, dod\n  admissions (275 rows): hadm_id PK, subject_id FK, admission_type, insurance, hospital_expire_flag (1=died)\n  diagnoses_icd (4,506 rows): seq_num=1 is primary diagnosis, icd_version is 9 or 10\n  labevents (107,727 rows): itemid FK, value, valuenum, flag='abnormal' means outside reference range\n  prescriptions (18,087 rows): drug, dose, route, starttime, stoptime\n  icustays (140 rows): stay_id PK, first_careunit, los (length of stay in days)\n  chartevents (668,862 rows): ICU vital signs and physiological measurements\n  + d_labitems, d_icd_diagnoses, d_items reference dictionaries + 18 more tables\nKey verified facts (from actual SQL on real database):\n  - 100 patients: 43 female, 57 male. Mean anchor age: 61.8 years\n  - 31 patients have a recorded date of death (dod is not empty)\n  - In-hospital mortality: 15 deaths / 275 admissions = 5.5%\n  - Average ICU length of stay: 3.68 days (140 stays total)\n  - 107,727 lab results; 37.4% flagged as abnormal (40,275 results)\n  - 18,087 prescriptions | 668,862 chartevents\n  - Admission types: EW EMER, URGENT, ELECTIVE, DIRECT EMER, OBSERVATION ADMIT\n\nCROSS-DATABASE: patient_customer_bridge (100 rows) maps retail customer_id to MIMIC subject_id.\n\nAnswer all questions using these verified facts only."
      },
      {
        "role": "user",
        "content": "What is the flag column in labevents?"
      },
      {
        "role": "assistant",
        "content": "flag='abnormal' means the lab result is outside the clinical reference range (ref_range_lower to ref_range_upper). 37.4% of all 107,727 lab results in MIMIC are flagged as abnormal."
      }
    ]
  },
  {
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant trained on two real relational databases.\n\nDATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):\nTables:\n  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)\n  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)\n  invoices (25,900 rows): invoice_no PK, customer_id FK, invoice_date, is_cancelled (1=return/refund)\n  invoice_items (54,873 rows): invoice_no FK, stock_code FK, quantity, unit_price, line_total\n  patient_customer_bridge (100 rows): links customer_id to MIMIC subject_id\nRelationships: invoice_items.invoice_no -> invoices | invoice_items.stock_code -> products | invoices.customer_id -> customers\nKey verified facts (from actual SQL on real database):\n  - 4,372 customers across 38 countries\n  - Confirmed order revenue (is_cancelled=0): £1,276,568.60\n  - Cancellation rate: 14.8% (3,836 of 25,900 invoices)\n  - Average confirmed order value: £100.52 per invoice\n  - Top 5 countries by revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194\n  - Date range: 2010-12-01 to 2011-12-09\n\nDATABASE 2 - MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):\nTables:\n  patients (100 rows): subject_id PK, gender M/F, anchor_age, anchor_year_group, dod\n  admissions (275 rows): hadm_id PK, subject_id FK, admission_type, insurance, hospital_expire_flag (1=died)\n  diagnoses_icd (4,506 rows): seq_num=1 is primary diagnosis, icd_version is 9 or 10\n  labevents (107,727 rows): itemid FK, value, valuenum, flag='abnormal' means outside reference range\n  prescriptions (18,087 rows): drug, dose, route, starttime, stoptime\n  icustays (140 rows): stay_id PK, first_careunit, los (length of stay in days)\n  chartevents (668,862 rows): ICU vital signs and physiological measurements\n  + d_labitems, d_icd_diagnoses, d_items reference dictionaries + 18 more tables\nKey verified facts (from actual SQL on real database):\n  - 100 patients: 43 female, 57 male. Mean anchor age: 61.8 years\n  - 31 patients have a recorded date of death (dod is not empty)\n  - In-hospital mortality: 15 deaths / 275 admissions = 5.5%\n  - Average ICU length of stay: 3.68 days (140 stays total)\n  - 107,727 lab results; 37.4% flagged as abnormal (40,275 results)\n  - 18,087 prescriptions | 668,862 chartevents\n  - Admission types: EW EMER, URGENT, ELECTIVE, DIRECT EMER, OBSERVATION ADMIT\n\nCROSS-DATABASE: patient_customer_bridge (100 rows) maps retail customer_id to MIMIC subject_id.\n\nAnswer all questions using these verified facts only."
      },
      {
        "role": "user",
        "content": "How many lab results are in MIMIC and what proportion are abnormal?"
      },
      {
        "role": "assistant",
        "content": "107,727 laboratory results total. 40,275 (37.4%) are flagged as abnormal."
      }
    ]
  },
  {
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant trained on two real relational databases.\n\nDATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):\nTables:\n  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)\n  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)\n  invoices (25,900 rows): invoice_no PK, customer_id FK, invoice_date, is_cancelled (1=return/refund)\n  invoice_items (54,873 rows): invoice_no FK, stock_code FK, quantity, unit_price, line_total\n  patient_customer_bridge (100 rows): links customer_id to MIMIC subject_id\nRelationships: invoice_items.invoice_no -> invoices | invoice_items.stock_code -> products | invoices.customer_id -> customers\nKey verified facts (from actual SQL on real database):\n  - 4,372 customers across 38 countries\n  - Confirmed order revenue (is_cancelled=0): £1,276,568.60\n  - Cancellation rate: 14.8% (3,836 of 25,900 invoices)\n  - Average confirmed order value: £100.52 per invoice\n  - Top 5 countries by revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194\n  - Date range: 2010-12-01 to 2011-12-09\n\nDATABASE 2 - MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):\nTables:\n  patients (100 rows): subject_id PK, gender M/F, anchor_age, anchor_year_group, dod\n  admissions (275 rows): hadm_id PK, subject_id FK, admission_type, insurance, hospital_expire_flag (1=died)\n  diagnoses_icd (4,506 rows): seq_num=1 is primary diagnosis, icd_version is 9 or 10\n  labevents (107,727 rows): itemid FK, value, valuenum, flag='abnormal' means outside reference range\n  prescriptions (18,087 rows): drug, dose, route, starttime, stoptime\n  icustays (140 rows): stay_id PK, first_careunit, los (length of stay in days)\n  chartevents (668,862 rows): ICU vital signs and physiological measurements\n  + d_labitems, d_icd_diagnoses, d_items reference dictionaries + 18 more tables\nKey verified facts (from actual SQL on real database):\n  - 100 patients: 43 female, 57 male. Mean anchor age: 61.8 years\n  - 31 patients have a recorded date of death (dod is not empty)\n  - In-hospital mortality: 15 deaths / 275 admissions = 5.5%\n  - Average ICU length of stay: 3.68 days (140 stays total)\n  - 107,727 lab results; 37.4% flagged as abnormal (40,275 results)\n  - 18,087 prescriptions | 668,862 chartevents\n  - Admission types: EW EMER, URGENT, ELECTIVE, DIRECT EMER, OBSERVATION ADMIT\n\nCROSS-DATABASE: patient_customer_bridge (100 rows) maps retail customer_id to MIMIC subject_id.\n\nAnswer all questions using these verified facts only."
      },
      {
        "role": "user",
        "content": "What is the in-hospital mortality rate in MIMIC?"
      },
      {
        "role": "assistant",
        "content": "15 out of 275 hospital admissions resulted in death: in-hospital mortality rate = 5.5%."
      }
    ]
  },
  {
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant trained on two real relational databases.\n\nDATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):\nTables:\n  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)\n  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)\n  invoices (25,900 rows): invoice_no PK, customer_id FK, invoice_date, is_cancelled (1=return/refund)\n  invoice_items (54,873 rows): invoice_no FK, stock_code FK, quantity, unit_price, line_total\n  patient_customer_bridge (100 rows): links customer_id to MIMIC subject_id\nRelationships: invoice_items.invoice_no -> invoices | invoice_items.stock_code -> products | invoices.customer_id -> customers\nKey verified facts (from actual SQL on real database):\n  - 4,372 customers across 38 countries\n  - Confirmed order revenue (is_cancelled=0): £1,276,568.60\n  - Cancellation rate: 14.8% (3,836 of 25,900 invoices)\n  - Average confirmed order value: £100.52 per invoice\n  - Top 5 countries by revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194\n  - Date range: 2010-12-01 to 2011-12-09\n\nDATABASE 2 - MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):\nTables:\n  patients (100 rows): subject_id PK, gender M/F, anchor_age, anchor_year_group, dod\n  admissions (275 rows): hadm_id PK, subject_id FK, admission_type, insurance, hospital_expire_flag (1=died)\n  diagnoses_icd (4,506 rows): seq_num=1 is primary diagnosis, icd_version is 9 or 10\n  labevents (107,727 rows): itemid FK, value, valuenum, flag='abnormal' means outside reference range\n  prescriptions (18,087 rows): drug, dose, route, starttime, stoptime\n  icustays (140 rows): stay_id PK, first_careunit, los (length of stay in days)\n  chartevents (668,862 rows): ICU vital signs and physiological measurements\n  + d_labitems, d_icd_diagnoses, d_items reference dictionaries + 18 more tables\nKey verified facts (from actual SQL on real database):\n  - 100 patients: 43 female, 57 male. Mean anchor age: 61.8 years\n  - 31 patients have a recorded date of death (dod is not empty)\n  - In-hospital mortality: 15 deaths / 275 admissions = 5.5%\n  - Average ICU length of stay: 3.68 days (140 stays total)\n  - 107,727 lab results; 37.4% flagged as abnormal (40,275 results)\n  - 18,087 prescriptions | 668,862 chartevents\n  - Admission types: EW EMER, URGENT, ELECTIVE, DIRECT EMER, OBSERVATION ADMIT\n\nCROSS-DATABASE: patient_customer_bridge (100 rows) maps retail customer_id to MIMIC subject_id.\n\nAnswer all questions using these verified facts only."
      },
      {
        "role": "user",
        "content": "Explain the full schema of the Online Retail database with relationships."
      },
      {
        "role": "assistant",
        "content": "Four core tables: customers (customer_id PK, country, segment) -> invoices (invoice_no PK, customer_id FK, invoice_date, is_cancelled) -> invoice_items (invoice_no FK, stock_code FK, quantity, unit_price, line_total) -> products (stock_code PK, description, category, unit_price). Plus patient_customer_bridge linking customer_id to MIMIC subject_id."
      }
    ]
  },
  {
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant trained on two real relational databases.\n\nDATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):\nTables:\n  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)\n  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)\n  invoices (25,900 rows): invoice_no PK, customer_id FK, invoice_date, is_cancelled (1=return/refund)\n  invoice_items (54,873 rows): invoice_no FK, stock_code FK, quantity, unit_price, line_total\n  patient_customer_bridge (100 rows): links customer_id to MIMIC subject_id\nRelationships: invoice_items.invoice_no -> invoices | invoice_items.stock_code -> products | invoices.customer_id -> customers\nKey verified facts (from actual SQL on real database):\n  - 4,372 customers across 38 countries\n  - Confirmed order revenue (is_cancelled=0): £1,276,568.60\n  - Cancellation rate: 14.8% (3,836 of 25,900 invoices)\n  - Average confirmed order value: £100.52 per invoice\n  - Top 5 countries by revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194\n  - Date range: 2010-12-01 to 2011-12-09\n\nDATABASE 2 - MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):\nTables:\n  patients (100 rows): subject_id PK, gender M/F, anchor_age, anchor_year_group, dod\n  admissions (275 rows): hadm_id PK, subject_id FK, admission_type, insurance, hospital_expire_flag (1=died)\n  diagnoses_icd (4,506 rows): seq_num=1 is primary diagnosis, icd_version is 9 or 10\n  labevents (107,727 rows): itemid FK, value, valuenum, flag='abnormal' means outside reference range\n  prescriptions (18,087 rows): drug, dose, route, starttime, stoptime\n  icustays (140 rows): stay_id PK, first_careunit, los (length of stay in days)\n  chartevents (668,862 rows): ICU vital signs and physiological measurements\n  + d_labitems, d_icd_diagnoses, d_items reference dictionaries + 18 more tables\nKey verified facts (from actual SQL on real database):\n  - 100 patients: 43 female, 57 male. Mean anchor age: 61.8 years\n  - 31 patients have a recorded date of death (dod is not empty)\n  - In-hospital mortality: 15 deaths / 275 admissions = 5.5%\n  - Average ICU length of stay: 3.68 days (140 stays total)\n  - 107,727 lab results; 37.4% flagged as abnormal (40,275 results)\n  - 18,087 prescriptions | 668,862 chartevents\n  - Admission types: EW EMER, URGENT, ELECTIVE, DIRECT EMER, OBSERVATION ADMIT\n\nCROSS-DATABASE: patient_customer_bridge (100 rows) maps retail customer_id to MIMIC subject_id.\n\nAnswer all questions using these verified facts only."
      },
      {
        "role": "user",
        "content": "How many hospital admissions are in MIMIC?"
      },
      {
        "role": "assistant",
        "content": "275 hospital admissions. 15 resulted in in-hospital death (5.5% mortality rate)."
      }
    ]
  },
  {
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant trained on two real relational databases.\n\nDATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):\nTables:\n  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)\n  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)\n  invoices (25,900 rows): invoice_no PK, customer_id FK, invoice_date, is_cancelled (1=return/refund)\n  invoice_items (54,873 rows): invoice_no FK, stock_code FK, quantity, unit_price, line_total\n  patient_customer_bridge (100 rows): links customer_id to MIMIC subject_id\nRelationships: invoice_items.invoice_no -> invoices | invoice_items.stock_code -> products | invoices.customer_id -> customers\nKey verified facts (from actual SQL on real database):\n  - 4,372 customers across 38 countries\n  - Confirmed order revenue (is_cancelled=0): £1,276,568.60\n  - Cancellation rate: 14.8% (3,836 of 25,900 invoices)\n  - Average confirmed order value: £100.52 per invoice\n  - Top 5 countries by revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194\n  - Date range: 2010-12-01 to 2011-12-09\n\nDATABASE 2 - MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):\nTables:\n  patients (100 rows): subject_id PK, gender M/F, anchor_age, anchor_year_group, dod\n  admissions (275 rows): hadm_id PK, subject_id FK, admission_type, insurance, hospital_expire_flag (1=died)\n  diagnoses_icd (4,506 rows): seq_num=1 is primary diagnosis, icd_version is 9 or 10\n  labevents (107,727 rows): itemid FK, value, valuenum, flag='abnormal' means outside reference range\n  prescriptions (18,087 rows): drug, dose, route, starttime, stoptime\n  icustays (140 rows): stay_id PK, first_careunit, los (length of stay in days)\n  chartevents (668,862 rows): ICU vital signs and physiological measurements\n  + d_labitems, d_icd_diagnoses, d_items reference dictionaries + 18 more tables\nKey verified facts (from actual SQL on real database):\n  - 100 patients: 43 female, 57 male. Mean anchor age: 61.8 years\n  - 31 patients have a recorded date of death (dod is not empty)\n  - In-hospital mortality: 15 deaths / 275 admissions = 5.5%\n  - Average ICU length of stay: 3.68 days (140 stays total)\n  - 107,727 lab results; 37.4% flagged as abnormal (40,275 results)\n  - 18,087 prescriptions | 668,862 chartevents\n  - Admission types: EW EMER, URGENT, ELECTIVE, DIRECT EMER, OBSERVATION ADMIT\n\nCROSS-DATABASE: patient_customer_bridge (100 rows) maps retail customer_id to MIMIC subject_id.\n\nAnswer all questions using these verified facts only."
      },
      {
        "role": "user",
        "content": "What is the total confirmed revenue from the retail database?"
      },
      {
        "role": "assistant",
        "content": "Confirmed order revenue (is_cancelled=0): £1,276,568.60. Average order value: £100.52."
      }
    ]
  },
  {
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant trained on two real relational databases.\n\nDATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):\nTables:\n  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)\n  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)\n  invoices (25,900 rows): invoice_no PK, customer_id FK, invoice_date, is_cancelled (1=return/refund)\n  invoice_items (54,873 rows): invoice_no FK, stock_code FK, quantity, unit_price, line_total\n  patient_customer_bridge (100 rows): links customer_id to MIMIC subject_id\nRelationships: invoice_items.invoice_no -> invoices | invoice_items.stock_code -> products | invoices.customer_id -> customers\nKey verified facts (from actual SQL on real database):\n  - 4,372 customers across 38 countries\n  - Confirmed order revenue (is_cancelled=0): £1,276,568.60\n  - Cancellation rate: 14.8% (3,836 of 25,900 invoices)\n  - Average confirmed order value: £100.52 per invoice\n  - Top 5 countries by revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194\n  - Date range: 2010-12-01 to 2011-12-09\n\nDATABASE 2 - MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):\nTables:\n  patients (100 rows): subject_id PK, gender M/F, anchor_age, anchor_year_group, dod\n  admissions (275 rows): hadm_id PK, subject_id FK, admission_type, insurance, hospital_expire_flag (1=died)\n  diagnoses_icd (4,506 rows): seq_num=1 is primary diagnosis, icd_version is 9 or 10\n  labevents (107,727 rows): itemid FK, value, valuenum, flag='abnormal' means outside reference range\n  prescriptions (18,087 rows): drug, dose, route, starttime, stoptime\n  icustays (140 rows): stay_id PK, first_careunit, los (length of stay in days)\n  chartevents (668,862 rows): ICU vital signs and physiological measurements\n  + d_labitems, d_icd_diagnoses, d_items reference dictionaries + 18 more tables\nKey verified facts (from actual SQL on real database):\n  - 100 patients: 43 female, 57 male. Mean anchor age: 61.8 years\n  - 31 patients have a recorded date of death (dod is not empty)\n  - In-hospital mortality: 15 deaths / 275 admissions = 5.5%\n  - Average ICU length of stay: 3.68 days (140 stays total)\n  - 107,727 lab results; 37.4% flagged as abnormal (40,275 results)\n  - 18,087 prescriptions | 668,862 chartevents\n  - Admission types: EW EMER, URGENT, ELECTIVE, DIRECT EMER, OBSERVATION ADMIT\n\nCROSS-DATABASE: patient_customer_bridge (100 rows) maps retail customer_id to MIMIC subject_id.\n\nAnswer all questions using these verified facts only."
      },
      {
        "role": "user",
        "content": "How many ICU measurements are in chartevents?"
      },
      {
        "role": "assistant",
        "content": "668,862 clinical measurements (vital signs, physiological parameters) across all 140 ICU stays."
      }
    ]
  },
  {
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant trained on two real relational databases.\n\nDATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):\nTables:\n  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)\n  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)\n  invoices (25,900 rows): invoice_no PK, customer_id FK, invoice_date, is_cancelled (1=return/refund)\n  invoice_items (54,873 rows): invoice_no FK, stock_code FK, quantity, unit_price, line_total\n  patient_customer_bridge (100 rows): links customer_id to MIMIC subject_id\nRelationships: invoice_items.invoice_no -> invoices | invoice_items.stock_code -> products | invoices.customer_id -> customers\nKey verified facts (from actual SQL on real database):\n  - 4,372 customers across 38 countries\n  - Confirmed order revenue (is_cancelled=0): £1,276,568.60\n  - Cancellation rate: 14.8% (3,836 of 25,900 invoices)\n  - Average confirmed order value: £100.52 per invoice\n  - Top 5 countries by revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194\n  - Date range: 2010-12-01 to 2011-12-09\n\nDATABASE 2 - MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):\nTables:\n  patients (100 rows): subject_id PK, gender M/F, anchor_age, anchor_year_group, dod\n  admissions (275 rows): hadm_id PK, subject_id FK, admission_type, insurance, hospital_expire_flag (1=died)\n  diagnoses_icd (4,506 rows): seq_num=1 is primary diagnosis, icd_version is 9 or 10\n  labevents (107,727 rows): itemid FK, value, valuenum, flag='abnormal' means outside reference range\n  prescriptions (18,087 rows): drug, dose, route, starttime, stoptime\n  icustays (140 rows): stay_id PK, first_careunit, los (length of stay in days)\n  chartevents (668,862 rows): ICU vital signs and physiological measurements\n  + d_labitems, d_icd_diagnoses, d_items reference dictionaries + 18 more tables\nKey verified facts (from actual SQL on real database):\n  - 100 patients: 43 female, 57 male. Mean anchor age: 61.8 years\n  - 31 patients have a recorded date of death (dod is not empty)\n  - In-hospital mortality: 15 deaths / 275 admissions = 5.5%\n  - Average ICU length of stay: 3.68 days (140 stays total)\n  - 107,727 lab results; 37.4% flagged as abnormal (40,275 results)\n  - 18,087 prescriptions | 668,862 chartevents\n  - Admission types: EW EMER, URGENT, ELECTIVE, DIRECT EMER, OBSERVATION ADMIT\n\nCROSS-DATABASE: patient_customer_bridge (100 rows) maps retail customer_id to MIMIC subject_id.\n\nAnswer all questions using these verified facts only."
      },
      {
        "role": "user",
        "content": "What does line_total represent in invoice_items?"
      },
      {
        "role": "assistant",
        "content": "line_total = quantity x unit_price for that line item, in British pounds GBP. Sum line_total across confirmed invoices (is_cancelled=0) to get total confirmed revenue."
      }
    ]
  },
  {
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant trained on two real relational databases.\n\nDATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):\nTables:\n  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)\n  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)\n  invoices (25,900 rows): invoice_no PK, customer_id FK, invoice_date, is_cancelled (1=return/refund)\n  invoice_items (54,873 rows): invoice_no FK, stock_code FK, quantity, unit_price, line_total\n  patient_customer_bridge (100 rows): links customer_id to MIMIC subject_id\nRelationships: invoice_items.invoice_no -> invoices | invoice_items.stock_code -> products | invoices.customer_id -> customers\nKey verified facts (from actual SQL on real database):\n  - 4,372 customers across 38 countries\n  - Confirmed order revenue (is_cancelled=0): £1,276,568.60\n  - Cancellation rate: 14.8% (3,836 of 25,900 invoices)\n  - Average confirmed order value: £100.52 per invoice\n  - Top 5 countries by revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194\n  - Date range: 2010-12-01 to 2011-12-09\n\nDATABASE 2 - MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):\nTables:\n  patients (100 rows): subject_id PK, gender M/F, anchor_age, anchor_year_group, dod\n  admissions (275 rows): hadm_id PK, subject_id FK, admission_type, insurance, hospital_expire_flag (1=died)\n  diagnoses_icd (4,506 rows): seq_num=1 is primary diagnosis, icd_version is 9 or 10\n  labevents (107,727 rows): itemid FK, value, valuenum, flag='abnormal' means outside reference range\n  prescriptions (18,087 rows): drug, dose, route, starttime, stoptime\n  icustays (140 rows): stay_id PK, first_careunit, los (length of stay in days)\n  chartevents (668,862 rows): ICU vital signs and physiological measurements\n  + d_labitems, d_icd_diagnoses, d_items reference dictionaries + 18 more tables\nKey verified facts (from actual SQL on real database):\n  - 100 patients: 43 female, 57 male. Mean anchor age: 61.8 years\n  - 31 patients have a recorded date of death (dod is not empty)\n  - In-hospital mortality: 15 deaths / 275 admissions = 5.5%\n  - Average ICU length of stay: 3.68 days (140 stays total)\n  - 107,727 lab results; 37.4% flagged as abnormal (40,275 results)\n  - 18,087 prescriptions | 668,862 chartevents\n  - Admission types: EW EMER, URGENT, ELECTIVE, DIRECT EMER, OBSERVATION ADMIT\n\nCROSS-DATABASE: patient_customer_bridge (100 rows) maps retail customer_id to MIMIC subject_id.\n\nAnswer all questions using these verified facts only."
      },
      {
        "role": "user",
        "content": "What does the 5.5% mortality rate indicate about the MIMIC patients?"
      },
      {
        "role": "assistant",
        "content": "15 of 275 admissions ended in in-hospital death. ICU patients are high-acuity by definition. Small demo sample of 100 patients limits statistical generalisability."
      }
    ]
  },
  {
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant trained on two real relational databases.\n\nDATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):\nTables:\n  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)\n  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)\n  invoices (25,900 rows): invoice_no PK, customer_id FK, invoice_date, is_cancelled (1=return/refund)\n  invoice_items (54,873 rows): invoice_no FK, stock_code FK, quantity, unit_price, line_total\n  patient_customer_bridge (100 rows): links customer_id to MIMIC subject_id\nRelationships: invoice_items.invoice_no -> invoices | invoice_items.stock_code -> products | invoices.customer_id -> customers\nKey verified facts (from actual SQL on real database):\n  - 4,372 customers across 38 countries\n  - Confirmed order revenue (is_cancelled=0): £1,276,568.60\n  - Cancellation rate: 14.8% (3,836 of 25,900 invoices)\n  - Average confirmed order value: £100.52 per invoice\n  - Top 5 countries by revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194\n  - Date range: 2010-12-01 to 2011-12-09\n\nDATABASE 2 - MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):\nTables:\n  patients (100 rows): subject_id PK, gender M/F, anchor_age, anchor_year_group, dod\n  admissions (275 rows): hadm_id PK, subject_id FK, admission_type, insurance, hospital_expire_flag (1=died)\n  diagnoses_icd (4,506 rows): seq_num=1 is primary diagnosis, icd_version is 9 or 10\n  labevents (107,727 rows): itemid FK, value, valuenum, flag='abnormal' means outside reference range\n  prescriptions (18,087 rows): drug, dose, route, starttime, stoptime\n  icustays (140 rows): stay_id PK, first_careunit, los (length of stay in days)\n  chartevents (668,862 rows): ICU vital signs and physiological measurements\n  + d_labitems, d_icd_diagnoses, d_items reference dictionaries + 18 more tables\nKey verified facts (from actual SQL on real database):\n  - 100 patients: 43 female, 57 male. Mean anchor age: 61.8 years\n  - 31 patients have a recorded date of death (dod is not empty)\n  - In-hospital mortality: 15 deaths / 275 admissions = 5.5%\n  - Average ICU length of stay: 3.68 days (140 stays total)\n  - 107,727 lab results; 37.4% flagged as abnormal (40,275 results)\n  - 18,087 prescriptions | 668,862 chartevents\n  - Admission types: EW EMER, URGENT, ELECTIVE, DIRECT EMER, OBSERVATION ADMIT\n\nCROSS-DATABASE: patient_customer_bridge (100 rows) maps retail customer_id to MIMIC subject_id.\n\nAnswer all questions using these verified facts only."
      },
      {
        "role": "user",
        "content": "How many products are in the retail catalogue?"
      },
      {
        "role": "assistant",
        "content": "1,124 distinct products identified by unique stock codes."
      }
    ]
  },
  {
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant trained on two real relational databases.\n\nDATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):\nTables:\n  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)\n  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)\n  invoices (25,900 rows): invoice_no PK, customer_id FK, invoice_date, is_cancelled (1=return/refund)\n  invoice_items (54,873 rows): invoice_no FK, stock_code FK, quantity, unit_price, line_total\n  patient_customer_bridge (100 rows): links customer_id to MIMIC subject_id\nRelationships: invoice_items.invoice_no -> invoices | invoice_items.stock_code -> products | invoices.customer_id -> customers\nKey verified facts (from actual SQL on real database):\n  - 4,372 customers across 38 countries\n  - Confirmed order revenue (is_cancelled=0): £1,276,568.60\n  - Cancellation rate: 14.8% (3,836 of 25,900 invoices)\n  - Average confirmed order value: £100.52 per invoice\n  - Top 5 countries by revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194\n  - Date range: 2010-12-01 to 2011-12-09\n\nDATABASE 2 - MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):\nTables:\n  patients (100 rows): subject_id PK, gender M/F, anchor_age, anchor_year_group, dod\n  admissions (275 rows): hadm_id PK, subject_id FK, admission_type, insurance, hospital_expire_flag (1=died)\n  diagnoses_icd (4,506 rows): seq_num=1 is primary diagnosis, icd_version is 9 or 10\n  labevents (107,727 rows): itemid FK, value, valuenum, flag='abnormal' means outside reference range\n  prescriptions (18,087 rows): drug, dose, route, starttime, stoptime\n  icustays (140 rows): stay_id PK, first_careunit, los (length of stay in days)\n  chartevents (668,862 rows): ICU vital signs and physiological measurements\n  + d_labitems, d_icd_diagnoses, d_items reference dictionaries + 18 more tables\nKey verified facts (from actual SQL on real database):\n  - 100 patients: 43 female, 57 male. Mean anchor age: 61.8 years\n  - 31 patients have a recorded date of death (dod is not empty)\n  - In-hospital mortality: 15 deaths / 275 admissions = 5.5%\n  - Average ICU length of stay: 3.68 days (140 stays total)\n  - 107,727 lab results; 37.4% flagged as abnormal (40,275 results)\n  - 18,087 prescriptions | 668,862 chartevents\n  - Admission types: EW EMER, URGENT, ELECTIVE, DIRECT EMER, OBSERVATION ADMIT\n\nCROSS-DATABASE: patient_customer_bridge (100 rows) maps retail customer_id to MIMIC subject_id.\n\nAnswer all questions using these verified facts only."
      },
      {
        "role": "user",
        "content": "How are the retail and MIMIC databases linked?"
      },
      {
        "role": "assistant",
        "content": "Via patient_customer_bridge (100 rows) which maps retail customer_id to MIMIC subject_id. Exists in both databases, enabling combined retail + clinical analysis."
      }
    ]
  },
  {
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant trained on two real relational databases.\n\nDATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):\nTables:\n  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)\n  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)\n  invoices (25,900 rows): invoice_no PK, customer_id FK, invoice_date, is_cancelled (1=return/refund)\n  invoice_items (54,873 rows): invoice_no FK, stock_code FK, quantity, unit_price, line_total\n  patient_customer_bridge (100 rows): links customer_id to MIMIC subject_id\nRelationships: invoice_items.invoice_no -> invoices | invoice_items.stock_code -> products | invoices.customer_id -> customers\nKey verified facts (from actual SQL on real database):\n  - 4,372 customers across 38 countries\n  - Confirmed order revenue (is_cancelled=0): £1,276,568.60\n  - Cancellation rate: 14.8% (3,836 of 25,900 invoices)\n  - Average confirmed order value: £100.52 per invoice\n  - Top 5 countries by revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194\n  - Date range: 2010-12-01 to 2011-12-09\n\nDATABASE 2 - MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):\nTables:\n  patients (100 rows): subject_id PK, gender M/F, anchor_age, anchor_year_group, dod\n  admissions (275 rows): hadm_id PK, subject_id FK, admission_type, insurance, hospital_expire_flag (1=died)\n  diagnoses_icd (4,506 rows): seq_num=1 is primary diagnosis, icd_version is 9 or 10\n  labevents (107,727 rows): itemid FK, value, valuenum, flag='abnormal' means outside reference range\n  prescriptions (18,087 rows): drug, dose, route, starttime, stoptime\n  icustays (140 rows): stay_id PK, first_careunit, los (length of stay in days)\n  chartevents (668,862 rows): ICU vital signs and physiological measurements\n  + d_labitems, d_icd_diagnoses, d_items reference dictionaries + 18 more tables\nKey verified facts (from actual SQL on real database):\n  - 100 patients: 43 female, 57 male. Mean anchor age: 61.8 years\n  - 31 patients have a recorded date of death (dod is not empty)\n  - In-hospital mortality: 15 deaths / 275 admissions = 5.5%\n  - Average ICU length of stay: 3.68 days (140 stays total)\n  - 107,727 lab results; 37.4% flagged as abnormal (40,275 results)\n  - 18,087 prescriptions | 668,862 chartevents\n  - Admission types: EW EMER, URGENT, ELECTIVE, DIRECT EMER, OBSERVATION ADMIT\n\nCROSS-DATABASE: patient_customer_bridge (100 rows) maps retail customer_id to MIMIC subject_id.\n\nAnswer all questions using these verified facts only."
      },
      {
        "role": "user",
        "content": "How many MIMIC patients have a recorded date of death?"
      },
      {
        "role": "assistant",
        "content": "31 patients have a recorded date of death (dod field not empty). 69 have no dod recorded."
      }
    ]
  },
  {
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant trained on two real relational databases.\n\nDATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):\nTables:\n  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)\n  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)\n  invoices (25,900 rows): invoice_no PK, customer_id FK, invoice_date, is_cancelled (1=return/refund)\n  invoice_items (54,873 rows): invoice_no FK, stock_code FK, quantity, unit_price, line_total\n  patient_customer_bridge (100 rows): links customer_id to MIMIC subject_id\nRelationships: invoice_items.invoice_no -> invoices | invoice_items.stock_code -> products | invoices.customer_id -> customers\nKey verified facts (from actual SQL on real database):\n  - 4,372 customers across 38 countries\n  - Confirmed order revenue (is_cancelled=0): £1,276,568.60\n  - Cancellation rate: 14.8% (3,836 of 25,900 invoices)\n  - Average confirmed order value: £100.52 per invoice\n  - Top 5 countries by revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194\n  - Date range: 2010-12-01 to 2011-12-09\n\nDATABASE 2 - MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):\nTables:\n  patients (100 rows): subject_id PK, gender M/F, anchor_age, anchor_year_group, dod\n  admissions (275 rows): hadm_id PK, subject_id FK, admission_type, insurance, hospital_expire_flag (1=died)\n  diagnoses_icd (4,506 rows): seq_num=1 is primary diagnosis, icd_version is 9 or 10\n  labevents (107,727 rows): itemid FK, value, valuenum, flag='abnormal' means outside reference range\n  prescriptions (18,087 rows): drug, dose, route, starttime, stoptime\n  icustays (140 rows): stay_id PK, first_careunit, los (length of stay in days)\n  chartevents (668,862 rows): ICU vital signs and physiological measurements\n  + d_labitems, d_icd_diagnoses, d_items reference dictionaries + 18 more tables\nKey verified facts (from actual SQL on real database):\n  - 100 patients: 43 female, 57 male. Mean anchor age: 61.8 years\n  - 31 patients have a recorded date of death (dod is not empty)\n  - In-hospital mortality: 15 deaths / 275 admissions = 5.5%\n  - Average ICU length of stay: 3.68 days (140 stays total)\n  - 107,727 lab results; 37.4% flagged as abnormal (40,275 results)\n  - 18,087 prescriptions | 668,862 chartevents\n  - Admission types: EW EMER, URGENT, ELECTIVE, DIRECT EMER, OBSERVATION ADMIT\n\nCROSS-DATABASE: patient_customer_bridge (100 rows) maps retail customer_id to MIMIC subject_id.\n\nAnswer all questions using these verified facts only."
      },
      {
        "role": "user",
        "content": "What tables are in the Online Retail database?"
      },
      {
        "role": "assistant",
        "content": "The Online Retail database has 5 tables: customers (4,372 rows) with customer profiles; products (1,124 rows) with stock codes, descriptions, and GBP prices; invoices (25,900 rows) as order headers with is_cancelled flag; invoice_items (54,873 rows) linking invoices to products with quantity and price; and patient_customer_bridge (100 rows) linking to MIMIC patients."
      }
    ]
  },
  {
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant trained on two real relational databases.\n\nDATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):\nTables:\n  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)\n  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)\n  invoices (25,900 rows): invoice_no PK, customer_id FK, invoice_date, is_cancelled (1=return/refund)\n  invoice_items (54,873 rows): invoice_no FK, stock_code FK, quantity, unit_price, line_total\n  patient_customer_bridge (100 rows): links customer_id to MIMIC subject_id\nRelationships: invoice_items.invoice_no -> invoices | invoice_items.stock_code -> products | invoices.customer_id -> customers\nKey verified facts (from actual SQL on real database):\n  - 4,372 customers across 38 countries\n  - Confirmed order revenue (is_cancelled=0): £1,276,568.60\n  - Cancellation rate: 14.8% (3,836 of 25,900 invoices)\n  - Average confirmed order value: £100.52 per invoice\n  - Top 5 countries by revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194\n  - Date range: 2010-12-01 to 2011-12-09\n\nDATABASE 2 - MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):\nTables:\n  patients (100 rows): subject_id PK, gender M/F, anchor_age, anchor_year_group, dod\n  admissions (275 rows): hadm_id PK, subject_id FK, admission_type, insurance, hospital_expire_flag (1=died)\n  diagnoses_icd (4,506 rows): seq_num=1 is primary diagnosis, icd_version is 9 or 10\n  labevents (107,727 rows): itemid FK, value, valuenum, flag='abnormal' means outside reference range\n  prescriptions (18,087 rows): drug, dose, route, starttime, stoptime\n  icustays (140 rows): stay_id PK, first_careunit, los (length of stay in days)\n  chartevents (668,862 rows): ICU vital signs and physiological measurements\n  + d_labitems, d_icd_diagnoses, d_items reference dictionaries + 18 more tables\nKey verified facts (from actual SQL on real database):\n  - 100 patients: 43 female, 57 male. Mean anchor age: 61.8 years\n  - 31 patients have a recorded date of death (dod is not empty)\n  - In-hospital mortality: 15 deaths / 275 admissions = 5.5%\n  - Average ICU length of stay: 3.68 days (140 stays total)\n  - 107,727 lab results; 37.4% flagged as abnormal (40,275 results)\n  - 18,087 prescriptions | 668,862 chartevents\n  - Admission types: EW EMER, URGENT, ELECTIVE, DIRECT EMER, OBSERVATION ADMIT\n\nCROSS-DATABASE: patient_customer_bridge (100 rows) maps retail customer_id to MIMIC subject_id.\n\nAnswer all questions using these verified facts only."
      },
      {
        "role": "user",
        "content": "What two databases are in this system?"
      },
      {
        "role": "assistant",
        "content": "1) Online Retail (retail.db): UK e-commerce Dec 2010-Dec 2011, 4,372 customers, 25,900 invoices, £1,276,568 confirmed revenue. 2) MIMIC-IV Clinical Demo (mimic.db): 100 ICU patients, 275 admissions, 5.5% mortality, 1.19M total rows. Linked via patient_customer_bridge (100 rows)."
      }
    ]
  },
  {
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant trained on two real relational databases.\n\nDATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):\nTables:\n  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)\n  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)\n  invoices (25,900 rows): invoice_no PK, customer_id FK, invoice_date, is_cancelled (1=return/refund)\n  invoice_items (54,873 rows): invoice_no FK, stock_code FK, quantity, unit_price, line_total\n  patient_customer_bridge (100 rows): links customer_id to MIMIC subject_id\nRelationships: invoice_items.invoice_no -> invoices | invoice_items.stock_code -> products | invoices.customer_id -> customers\nKey verified facts (from actual SQL on real database):\n  - 4,372 customers across 38 countries\n  - Confirmed order revenue (is_cancelled=0): £1,276,568.60\n  - Cancellation rate: 14.8% (3,836 of 25,900 invoices)\n  - Average confirmed order value: £100.52 per invoice\n  - Top 5 countries by revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194\n  - Date range: 2010-12-01 to 2011-12-09\n\nDATABASE 2 - MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):\nTables:\n  patients (100 rows): subject_id PK, gender M/F, anchor_age, anchor_year_group, dod\n  admissions (275 rows): hadm_id PK, subject_id FK, admission_type, insurance, hospital_expire_flag (1=died)\n  diagnoses_icd (4,506 rows): seq_num=1 is primary diagnosis, icd_version is 9 or 10\n  labevents (107,727 rows): itemid FK, value, valuenum, flag='abnormal' means outside reference range\n  prescriptions (18,087 rows): drug, dose, route, starttime, stoptime\n  icustays (140 rows): stay_id PK, first_careunit, los (length of stay in days)\n  chartevents (668,862 rows): ICU vital signs and physiological measurements\n  + d_labitems, d_icd_diagnoses, d_items reference dictionaries + 18 more tables\nKey verified facts (from actual SQL on real database):\n  - 100 patients: 43 female, 57 male. Mean anchor age: 61.8 years\n  - 31 patients have a recorded date of death (dod is not empty)\n  - In-hospital mortality: 15 deaths / 275 admissions = 5.5%\n  - Average ICU length of stay: 3.68 days (140 stays total)\n  - 107,727 lab results; 37.4% flagged as abnormal (40,275 results)\n  - 18,087 prescriptions | 668,862 chartevents\n  - Admission types: EW EMER, URGENT, ELECTIVE, DIRECT EMER, OBSERVATION ADMIT\n\nCROSS-DATABASE: patient_customer_bridge (100 rows) maps retail customer_id to MIMIC subject_id.\n\nAnswer all questions using these verified facts only."
      },
      {
        "role": "user",
        "content": "How many patients are in the MIMIC-IV demo?"
      },
      {
        "role": "assistant",
        "content": "100 patients: 43 female and 57 male, mean anchor age 61.8 years."
      }
    ]
  },
  {
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant trained on two real relational databases.\n\nDATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):\nTables:\n  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)\n  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)\n  invoices (25,900 rows): invoice_no PK, customer_id FK, invoice_date, is_cancelled (1=return/refund)\n  invoice_items (54,873 rows): invoice_no FK, stock_code FK, quantity, unit_price, line_total\n  patient_customer_bridge (100 rows): links customer_id to MIMIC subject_id\nRelationships: invoice_items.invoice_no -> invoices | invoice_items.stock_code -> products | invoices.customer_id -> customers\nKey verified facts (from actual SQL on real database):\n  - 4,372 customers across 38 countries\n  - Confirmed order revenue (is_cancelled=0): £1,276,568.60\n  - Cancellation rate: 14.8% (3,836 of 25,900 invoices)\n  - Average confirmed order value: £100.52 per invoice\n  - Top 5 countries by revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194\n  - Date range: 2010-12-01 to 2011-12-09\n\nDATABASE 2 - MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):\nTables:\n  patients (100 rows): subject_id PK, gender M/F, anchor_age, anchor_year_group, dod\n  admissions (275 rows): hadm_id PK, subject_id FK, admission_type, insurance, hospital_expire_flag (1=died)\n  diagnoses_icd (4,506 rows): seq_num=1 is primary diagnosis, icd_version is 9 or 10\n  labevents (107,727 rows): itemid FK, value, valuenum, flag='abnormal' means outside reference range\n  prescriptions (18,087 rows): drug, dose, route, starttime, stoptime\n  icustays (140 rows): stay_id PK, first_careunit, los (length of stay in days)\n  chartevents (668,862 rows): ICU vital signs and physiological measurements\n  + d_labitems, d_icd_diagnoses, d_items reference dictionaries + 18 more tables\nKey verified facts (from actual SQL on real database):\n  - 100 patients: 43 female, 57 male. Mean anchor age: 61.8 years\n  - 31 patients have a recorded date of death (dod is not empty)\n  - In-hospital mortality: 15 deaths / 275 admissions = 5.5%\n  - Average ICU length of stay: 3.68 days (140 stays total)\n  - 107,727 lab results; 37.4% flagged as abnormal (40,275 results)\n  - 18,087 prescriptions | 668,862 chartevents\n  - Admission types: EW EMER, URGENT, ELECTIVE, DIRECT EMER, OBSERVATION ADMIT\n\nCROSS-DATABASE: patient_customer_bridge (100 rows) maps retail customer_id to MIMIC subject_id.\n\nAnswer all questions using these verified facts only."
      },
      {
        "role": "user",
        "content": "How many invoices are in the retail database?"
      },
      {
        "role": "assistant",
        "content": "25,900 total invoices: 3,836 cancellations (14.8%) and 21,064 confirmed purchases."
      }
    ]
  },
  {
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant trained on two real relational databases.\n\nDATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):\nTables:\n  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)\n  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)\n  invoices (25,900 rows): invoice_no PK, customer_id FK, invoice_date, is_cancelled (1=return/refund)\n  invoice_items (54,873 rows): invoice_no FK, stock_code FK, quantity, unit_price, line_total\n  patient_customer_bridge (100 rows): links customer_id to MIMIC subject_id\nRelationships: invoice_items.invoice_no -> invoices | invoice_items.stock_code -> products | invoices.customer_id -> customers\nKey verified facts (from actual SQL on real database):\n  - 4,372 customers across 38 countries\n  - Confirmed order revenue (is_cancelled=0): £1,276,568.60\n  - Cancellation rate: 14.8% (3,836 of 25,900 invoices)\n  - Average confirmed order value: £100.52 per invoice\n  - Top 5 countries by revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194\n  - Date range: 2010-12-01 to 2011-12-09\n\nDATABASE 2 - MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):\nTables:\n  patients (100 rows): subject_id PK, gender M/F, anchor_age, anchor_year_group, dod\n  admissions (275 rows): hadm_id PK, subject_id FK, admission_type, insurance, hospital_expire_flag (1=died)\n  diagnoses_icd (4,506 rows): seq_num=1 is primary diagnosis, icd_version is 9 or 10\n  labevents (107,727 rows): itemid FK, value, valuenum, flag='abnormal' means outside reference range\n  prescriptions (18,087 rows): drug, dose, route, starttime, stoptime\n  icustays (140 rows): stay_id PK, first_careunit, los (length of stay in days)\n  chartevents (668,862 rows): ICU vital signs and physiological measurements\n  + d_labitems, d_icd_diagnoses, d_items reference dictionaries + 18 more tables\nKey verified facts (from actual SQL on real database):\n  - 100 patients: 43 female, 57 male. Mean anchor age: 61.8 years\n  - 31 patients have a recorded date of death (dod is not empty)\n  - In-hospital mortality: 15 deaths / 275 admissions = 5.5%\n  - Average ICU length of stay: 3.68 days (140 stays total)\n  - 107,727 lab results; 37.4% flagged as abnormal (40,275 results)\n  - 18,087 prescriptions | 668,862 chartevents\n  - Admission types: EW EMER, URGENT, ELECTIVE, DIRECT EMER, OBSERVATION ADMIT\n\nCROSS-DATABASE: patient_customer_bridge (100 rows) maps retail customer_id to MIMIC subject_id.\n\nAnswer all questions using these verified facts only."
      },
      {
        "role": "user",
        "content": "What does hospital_expire_flag mean?"
      },
      {
        "role": "assistant",
        "content": "hospital_expire_flag in the admissions table: 1 = patient died during that hospital admission (in-hospital death). 0 = patient survived and was discharged alive. Used to calculate in-hospital mortality rate."
      }
    ]
  },
  {
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant trained on two real relational databases.\n\nDATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):\nTables:\n  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)\n  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)\n  invoices (25,900 rows): invoice_no PK, customer_id FK, invoice_date, is_cancelled (1=return/refund)\n  invoice_items (54,873 rows): invoice_no FK, stock_code FK, quantity, unit_price, line_total\n  patient_customer_bridge (100 rows): links customer_id to MIMIC subject_id\nRelationships: invoice_items.invoice_no -> invoices | invoice_items.stock_code -> products | invoices.customer_id -> customers\nKey verified facts (from actual SQL on real database):\n  - 4,372 customers across 38 countries\n  - Confirmed order revenue (is_cancelled=0): £1,276,568.60\n  - Cancellation rate: 14.8% (3,836 of 25,900 invoices)\n  - Average confirmed order value: £100.52 per invoice\n  - Top 5 countries by revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194\n  - Date range: 2010-12-01 to 2011-12-09\n\nDATABASE 2 - MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):\nTables:\n  patients (100 rows): subject_id PK, gender M/F, anchor_age, anchor_year_group, dod\n  admissions (275 rows): hadm_id PK, subject_id FK, admission_type, insurance, hospital_expire_flag (1=died)\n  diagnoses_icd (4,506 rows): seq_num=1 is primary diagnosis, icd_version is 9 or 10\n  labevents (107,727 rows): itemid FK, value, valuenum, flag='abnormal' means outside reference range\n  prescriptions (18,087 rows): drug, dose, route, starttime, stoptime\n  icustays (140 rows): stay_id PK, first_careunit, los (length of stay in days)\n  chartevents (668,862 rows): ICU vital signs and physiological measurements\n  + d_labitems, d_icd_diagnoses, d_items reference dictionaries + 18 more tables\nKey verified facts (from actual SQL on real database):\n  - 100 patients: 43 female, 57 male. Mean anchor age: 61.8 years\n  - 31 patients have a recorded date of death (dod is not empty)\n  - In-hospital mortality: 15 deaths / 275 admissions = 5.5%\n  - Average ICU length of stay: 3.68 days (140 stays total)\n  - 107,727 lab results; 37.4% flagged as abnormal (40,275 results)\n  - 18,087 prescriptions | 668,862 chartevents\n  - Admission types: EW EMER, URGENT, ELECTIVE, DIRECT EMER, OBSERVATION ADMIT\n\nCROSS-DATABASE: patient_customer_bridge (100 rows) maps retail customer_id to MIMIC subject_id.\n\nAnswer all questions using these verified facts only."
      },
      {
        "role": "user",
        "content": "How many unique customers are in the retail database?"
      },
      {
        "role": "assistant",
        "content": "4,372 unique registered customers across 38 countries."
      }
    ]
  },
  {
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant trained on two real relational databases.\n\nDATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):\nTables:\n  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)\n  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)\n  invoices (25,900 rows): invoice_no PK, customer_id FK, invoice_date, is_cancelled (1=return/refund)\n  invoice_items (54,873 rows): invoice_no FK, stock_code FK, quantity, unit_price, line_total\n  patient_customer_bridge (100 rows): links customer_id to MIMIC subject_id\nRelationships: invoice_items.invoice_no -> invoices | invoice_items.stock_code -> products | invoices.customer_id -> customers\nKey verified facts (from actual SQL on real database):\n  - 4,372 customers across 38 countries\n  - Confirmed order revenue (is_cancelled=0): £1,276,568.60\n  - Cancellation rate: 14.8% (3,836 of 25,900 invoices)\n  - Average confirmed order value: £100.52 per invoice\n  - Top 5 countries by revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194\n  - Date range: 2010-12-01 to 2011-12-09\n\nDATABASE 2 - MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):\nTables:\n  patients (100 rows): subject_id PK, gender M/F, anchor_age, anchor_year_group, dod\n  admissions (275 rows): hadm_id PK, subject_id FK, admission_type, insurance, hospital_expire_flag (1=died)\n  diagnoses_icd (4,506 rows): seq_num=1 is primary diagnosis, icd_version is 9 or 10\n  labevents (107,727 rows): itemid FK, value, valuenum, flag='abnormal' means outside reference range\n  prescriptions (18,087 rows): drug, dose, route, starttime, stoptime\n  icustays (140 rows): stay_id PK, first_careunit, los (length of stay in days)\n  chartevents (668,862 rows): ICU vital signs and physiological measurements\n  + d_labitems, d_icd_diagnoses, d_items reference dictionaries + 18 more tables\nKey verified facts (from actual SQL on real database):\n  - 100 patients: 43 female, 57 male. Mean anchor age: 61.8 years\n  - 31 patients have a recorded date of death (dod is not empty)\n  - In-hospital mortality: 15 deaths / 275 admissions = 5.5%\n  - Average ICU length of stay: 3.68 days (140 stays total)\n  - 107,727 lab results; 37.4% flagged as abnormal (40,275 results)\n  - 18,087 prescriptions | 668,862 chartevents\n  - Admission types: EW EMER, URGENT, ELECTIVE, DIRECT EMER, OBSERVATION ADMIT\n\nCROSS-DATABASE: patient_customer_bridge (100 rows) maps retail customer_id to MIMIC subject_id.\n\nAnswer all questions using these verified facts only."
      },
      {
        "role": "user",
        "content": "How many prescriptions are in MIMIC?"
      },
      {
        "role": "assistant",
        "content": "18,087 prescription orders covering various medications."
      }
    ]
  },
  {
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant trained on two real relational databases.\n\nDATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):\nTables:\n  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)\n  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)\n  invoices (25,900 rows): invoice_no PK, customer_id FK, invoice_date, is_cancelled (1=return/refund)\n  invoice_items (54,873 rows): invoice_no FK, stock_code FK, quantity, unit_price, line_total\n  patient_customer_bridge (100 rows): links customer_id to MIMIC subject_id\nRelationships: invoice_items.invoice_no -> invoices | invoice_items.stock_code -> products | invoices.customer_id -> customers\nKey verified facts (from actual SQL on real database):\n  - 4,372 customers across 38 countries\n  - Confirmed order revenue (is_cancelled=0): £1,276,568.60\n  - Cancellation rate: 14.8% (3,836 of 25,900 invoices)\n  - Average confirmed order value: £100.52 per invoice\n  - Top 5 countries by revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194\n  - Date range: 2010-12-01 to 2011-12-09\n\nDATABASE 2 - MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):\nTables:\n  patients (100 rows): subject_id PK, gender M/F, anchor_age, anchor_year_group, dod\n  admissions (275 rows): hadm_id PK, subject_id FK, admission_type, insurance, hospital_expire_flag (1=died)\n  diagnoses_icd (4,506 rows): seq_num=1 is primary diagnosis, icd_version is 9 or 10\n  labevents (107,727 rows): itemid FK, value, valuenum, flag='abnormal' means outside reference range\n  prescriptions (18,087 rows): drug, dose, route, starttime, stoptime\n  icustays (140 rows): stay_id PK, first_careunit, los (length of stay in days)\n  chartevents (668,862 rows): ICU vital signs and physiological measurements\n  + d_labitems, d_icd_diagnoses, d_items reference dictionaries + 18 more tables\nKey verified facts (from actual SQL on real database):\n  - 100 patients: 43 female, 57 male. Mean anchor age: 61.8 years\n  - 31 patients have a recorded date of death (dod is not empty)\n  - In-hospital mortality: 15 deaths / 275 admissions = 5.5%\n  - Average ICU length of stay: 3.68 days (140 stays total)\n  - 107,727 lab results; 37.4% flagged as abnormal (40,275 results)\n  - 18,087 prescriptions | 668,862 chartevents\n  - Admission types: EW EMER, URGENT, ELECTIVE, DIRECT EMER, OBSERVATION ADMIT\n\nCROSS-DATABASE: patient_customer_bridge (100 rows) maps retail customer_id to MIMIC subject_id.\n\nAnswer all questions using these verified facts only."
      },
      {
        "role": "user",
        "content": "What does is_cancelled mean in the invoices table?"
      },
      {
        "role": "assistant",
        "content": "is_cancelled=1 means a return or refund (cancelled order). is_cancelled=0 means confirmed purchase. Cancelled invoices start with letter C in invoice_no. Always filter WHERE is_cancelled=0 for revenue calculations."
      }
    ]
  },
  {
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant trained on two real relational databases.\n\nDATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):\nTables:\n  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)\n  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)\n  invoices (25,900 rows): invoice_no PK, customer_id FK, invoice_date, is_cancelled (1=return/refund)\n  invoice_items (54,873 rows): invoice_no FK, stock_code FK, quantity, unit_price, line_total\n  patient_customer_bridge (100 rows): links customer_id to MIMIC subject_id\nRelationships: invoice_items.invoice_no -> invoices | invoice_items.stock_code -> products | invoices.customer_id -> customers\nKey verified facts (from actual SQL on real database):\n  - 4,372 customers across 38 countries\n  - Confirmed order revenue (is_cancelled=0): £1,276,568.60\n  - Cancellation rate: 14.8% (3,836 of 25,900 invoices)\n  - Average confirmed order value: £100.52 per invoice\n  - Top 5 countries by revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194\n  - Date range: 2010-12-01 to 2011-12-09\n\nDATABASE 2 - MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):\nTables:\n  patients (100 rows): subject_id PK, gender M/F, anchor_age, anchor_year_group, dod\n  admissions (275 rows): hadm_id PK, subject_id FK, admission_type, insurance, hospital_expire_flag (1=died)\n  diagnoses_icd (4,506 rows): seq_num=1 is primary diagnosis, icd_version is 9 or 10\n  labevents (107,727 rows): itemid FK, value, valuenum, flag='abnormal' means outside reference range\n  prescriptions (18,087 rows): drug, dose, route, starttime, stoptime\n  icustays (140 rows): stay_id PK, first_careunit, los (length of stay in days)\n  chartevents (668,862 rows): ICU vital signs and physiological measurements\n  + d_labitems, d_icd_diagnoses, d_items reference dictionaries + 18 more tables\nKey verified facts (from actual SQL on real database):\n  - 100 patients: 43 female, 57 male. Mean anchor age: 61.8 years\n  - 31 patients have a recorded date of death (dod is not empty)\n  - In-hospital mortality: 15 deaths / 275 admissions = 5.5%\n  - Average ICU length of stay: 3.68 days (140 stays total)\n  - 107,727 lab results; 37.4% flagged as abnormal (40,275 results)\n  - 18,087 prescriptions | 668,862 chartevents\n  - Admission types: EW EMER, URGENT, ELECTIVE, DIRECT EMER, OBSERVATION ADMIT\n\nCROSS-DATABASE: patient_customer_bridge (100 rows) maps retail customer_id to MIMIC subject_id.\n\nAnswer all questions using these verified facts only."
      },
      {
        "role": "user",
        "content": "How would you study cross-domain patterns using both databases?"
      },
      {
        "role": "assistant",
        "content": "Use patient_customer_bridge (100 links): find MIMIC patients meeting clinical criteria by subject_id, join to bridge for customer_ids, then query retail invoices for purchasing behaviour. Limited statistical power with 100 linked records."
      }
    ]
  },
  {
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant trained on two real relational databases.\n\nDATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):\nTables:\n  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)\n  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)\n  invoices (25,900 rows): invoice_no PK, customer_id FK, invoice_date, is_cancelled (1=return/refund)\n  invoice_items (54,873 rows): invoice_no FK, stock_code FK, quantity, unit_price, line_total\n  patient_customer_bridge (100 rows): links customer_id to MIMIC subject_id\nRelationships: invoice_items.invoice_no -> invoices | invoice_items.stock_code -> products | invoices.customer_id -> customers\nKey verified facts (from actual SQL on real database):\n  - 4,372 customers across 38 countries\n  - Confirmed order revenue (is_cancelled=0): £1,276,568.60\n  - Cancellation rate: 14.8% (3,836 of 25,900 invoices)\n  - Average confirmed order value: £100.52 per invoice\n  - Top 5 countries by revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194\n  - Date range: 2010-12-01 to 2011-12-09\n\nDATABASE 2 - MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):\nTables:\n  patients (100 rows): subject_id PK, gender M/F, anchor_age, anchor_year_group, dod\n  admissions (275 rows): hadm_id PK, subject_id FK, admission_type, insurance, hospital_expire_flag (1=died)\n  diagnoses_icd (4,506 rows): seq_num=1 is primary diagnosis, icd_version is 9 or 10\n  labevents (107,727 rows): itemid FK, value, valuenum, flag='abnormal' means outside reference range\n  prescriptions (18,087 rows): drug, dose, route, starttime, stoptime\n  icustays (140 rows): stay_id PK, first_careunit, los (length of stay in days)\n  chartevents (668,862 rows): ICU vital signs and physiological measurements\n  + d_labitems, d_icd_diagnoses, d_items reference dictionaries + 18 more tables\nKey verified facts (from actual SQL on real database):\n  - 100 patients: 43 female, 57 male. Mean anchor age: 61.8 years\n  - 31 patients have a recorded date of death (dod is not empty)\n  - In-hospital mortality: 15 deaths / 275 admissions = 5.5%\n  - Average ICU length of stay: 3.68 days (140 stays total)\n  - 107,727 lab results; 37.4% flagged as abnormal (40,275 results)\n  - 18,087 prescriptions | 668,862 chartevents\n  - Admission types: EW EMER, URGENT, ELECTIVE, DIRECT EMER, OBSERVATION ADMIT\n\nCROSS-DATABASE: patient_customer_bridge (100 rows) maps retail customer_id to MIMIC subject_id.\n\nAnswer all questions using these verified facts only."
      },
      {
        "role": "user",
        "content": "How are invoices and products linked?"
      },
      {
        "role": "assistant",
        "content": "Through the invoice_items table: invoice_items.invoice_no references invoices.invoice_no, and invoice_items.stock_code references products.stock_code. This many-to-many join means one invoice can contain multiple products."
      }
    ]
  },
  {
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant trained on two real relational databases.\n\nDATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):\nTables:\n  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)\n  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)\n  invoices (25,900 rows): invoice_no PK, customer_id FK, invoice_date, is_cancelled (1=return/refund)\n  invoice_items (54,873 rows): invoice_no FK, stock_code FK, quantity, unit_price, line_total\n  patient_customer_bridge (100 rows): links customer_id to MIMIC subject_id\nRelationships: invoice_items.invoice_no -> invoices | invoice_items.stock_code -> products | invoices.customer_id -> customers\nKey verified facts (from actual SQL on real database):\n  - 4,372 customers across 38 countries\n  - Confirmed order revenue (is_cancelled=0): £1,276,568.60\n  - Cancellation rate: 14.8% (3,836 of 25,900 invoices)\n  - Average confirmed order value: £100.52 per invoice\n  - Top 5 countries by revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194\n  - Date range: 2010-12-01 to 2011-12-09\n\nDATABASE 2 - MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):\nTables:\n  patients (100 rows): subject_id PK, gender M/F, anchor_age, anchor_year_group, dod\n  admissions (275 rows): hadm_id PK, subject_id FK, admission_type, insurance, hospital_expire_flag (1=died)\n  diagnoses_icd (4,506 rows): seq_num=1 is primary diagnosis, icd_version is 9 or 10\n  labevents (107,727 rows): itemid FK, value, valuenum, flag='abnormal' means outside reference range\n  prescriptions (18,087 rows): drug, dose, route, starttime, stoptime\n  icustays (140 rows): stay_id PK, first_careunit, los (length of stay in days)\n  chartevents (668,862 rows): ICU vital signs and physiological measurements\n  + d_labitems, d_icd_diagnoses, d_items reference dictionaries + 18 more tables\nKey verified facts (from actual SQL on real database):\n  - 100 patients: 43 female, 57 male. Mean anchor age: 61.8 years\n  - 31 patients have a recorded date of death (dod is not empty)\n  - In-hospital mortality: 15 deaths / 275 admissions = 5.5%\n  - Average ICU length of stay: 3.68 days (140 stays total)\n  - 107,727 lab results; 37.4% flagged as abnormal (40,275 results)\n  - 18,087 prescriptions | 668,862 chartevents\n  - Admission types: EW EMER, URGENT, ELECTIVE, DIRECT EMER, OBSERVATION ADMIT\n\nCROSS-DATABASE: patient_customer_bridge (100 rows) maps retail customer_id to MIMIC subject_id.\n\nAnswer all questions using these verified facts only."
      },
      {
        "role": "user",
        "content": "What does the 14.8% cancellation rate tell us about the retail business?"
      },
      {
        "role": "assistant",
        "content": "3,836 of 25,900 invoices are returns/cancellations. Revenue analysis must filter is_cancelled=0. Confirmed revenue is £1,276,568.60."
      }
    ]
  },
  {
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant trained on two real relational databases.\n\nDATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):\nTables:\n  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)\n  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)\n  invoices (25,900 rows): invoice_no PK, customer_id FK, invoice_date, is_cancelled (1=return/refund)\n  invoice_items (54,873 rows): invoice_no FK, stock_code FK, quantity, unit_price, line_total\n  patient_customer_bridge (100 rows): links customer_id to MIMIC subject_id\nRelationships: invoice_items.invoice_no -> invoices | invoice_items.stock_code -> products | invoices.customer_id -> customers\nKey verified facts (from actual SQL on real database):\n  - 4,372 customers across 38 countries\n  - Confirmed order revenue (is_cancelled=0): £1,276,568.60\n  - Cancellation rate: 14.8% (3,836 of 25,900 invoices)\n  - Average confirmed order value: £100.52 per invoice\n  - Top 5 countries by revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194\n  - Date range: 2010-12-01 to 2011-12-09\n\nDATABASE 2 - MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):\nTables:\n  patients (100 rows): subject_id PK, gender M/F, anchor_age, anchor_year_group, dod\n  admissions (275 rows): hadm_id PK, subject_id FK, admission_type, insurance, hospital_expire_flag (1=died)\n  diagnoses_icd (4,506 rows): seq_num=1 is primary diagnosis, icd_version is 9 or 10\n  labevents (107,727 rows): itemid FK, value, valuenum, flag='abnormal' means outside reference range\n  prescriptions (18,087 rows): drug, dose, route, starttime, stoptime\n  icustays (140 rows): stay_id PK, first_careunit, los (length of stay in days)\n  chartevents (668,862 rows): ICU vital signs and physiological measurements\n  + d_labitems, d_icd_diagnoses, d_items reference dictionaries + 18 more tables\nKey verified facts (from actual SQL on real database):\n  - 100 patients: 43 female, 57 male. Mean anchor age: 61.8 years\n  - 31 patients have a recorded date of death (dod is not empty)\n  - In-hospital mortality: 15 deaths / 275 admissions = 5.5%\n  - Average ICU length of stay: 3.68 days (140 stays total)\n  - 107,727 lab results; 37.4% flagged as abnormal (40,275 results)\n  - 18,087 prescriptions | 668,862 chartevents\n  - Admission types: EW EMER, URGENT, ELECTIVE, DIRECT EMER, OBSERVATION ADMIT\n\nCROSS-DATABASE: patient_customer_bridge (100 rows) maps retail customer_id to MIMIC subject_id.\n\nAnswer all questions using these verified facts only."
      },
      {
        "role": "user",
        "content": "What insurance types are in MIMIC admissions?"
      },
      {
        "role": "assistant",
        "content": "Medicare (elderly 65+), Medicaid (low-income), Other (private/self-pay). Medicare is most common in ICU patient populations due to age demographics."
      }
    ]
  },
  {
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant trained on two real relational databases.\n\nDATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):\nTables:\n  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)\n  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)\n  invoices (25,900 rows): invoice_no PK, customer_id FK, invoice_date, is_cancelled (1=return/refund)\n  invoice_items (54,873 rows): invoice_no FK, stock_code FK, quantity, unit_price, line_total\n  patient_customer_bridge (100 rows): links customer_id to MIMIC subject_id\nRelationships: invoice_items.invoice_no -> invoices | invoice_items.stock_code -> products | invoices.customer_id -> customers\nKey verified facts (from actual SQL on real database):\n  - 4,372 customers across 38 countries\n  - Confirmed order revenue (is_cancelled=0): £1,276,568.60\n  - Cancellation rate: 14.8% (3,836 of 25,900 invoices)\n  - Average confirmed order value: £100.52 per invoice\n  - Top 5 countries by revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194\n  - Date range: 2010-12-01 to 2011-12-09\n\nDATABASE 2 - MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):\nTables:\n  patients (100 rows): subject_id PK, gender M/F, anchor_age, anchor_year_group, dod\n  admissions (275 rows): hadm_id PK, subject_id FK, admission_type, insurance, hospital_expire_flag (1=died)\n  diagnoses_icd (4,506 rows): seq_num=1 is primary diagnosis, icd_version is 9 or 10\n  labevents (107,727 rows): itemid FK, value, valuenum, flag='abnormal' means outside reference range\n  prescriptions (18,087 rows): drug, dose, route, starttime, stoptime\n  icustays (140 rows): stay_id PK, first_careunit, los (length of stay in days)\n  chartevents (668,862 rows): ICU vital signs and physiological measurements\n  + d_labitems, d_icd_diagnoses, d_items reference dictionaries + 18 more tables\nKey verified facts (from actual SQL on real database):\n  - 100 patients: 43 female, 57 male. Mean anchor age: 61.8 years\n  - 31 patients have a recorded date of death (dod is not empty)\n  - In-hospital mortality: 15 deaths / 275 admissions = 5.5%\n  - Average ICU length of stay: 3.68 days (140 stays total)\n  - 107,727 lab results; 37.4% flagged as abnormal (40,275 results)\n  - 18,087 prescriptions | 668,862 chartevents\n  - Admission types: EW EMER, URGENT, ELECTIVE, DIRECT EMER, OBSERVATION ADMIT\n\nCROSS-DATABASE: patient_customer_bridge (100 rows) maps retail customer_id to MIMIC subject_id.\n\nAnswer all questions using these verified facts only."
      },
      {
        "role": "user",
        "content": "What does anchor_age mean in the MIMIC patients table?"
      },
      {
        "role": "assistant",
        "content": "anchor_age = patient's age in years at their anchor_year reference point. MIMIC shifts all dates for de-identification, so anchor_year is not the real calendar year. anchor_year_group gives a 3-year window (e.g. '2011 - 2013')."
      }
    ]
  },
  {
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant trained on two real relational databases.\n\nDATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):\nTables:\n  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)\n  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)\n  invoices (25,900 rows): invoice_no PK, customer_id FK, invoice_date, is_cancelled (1=return/refund)\n  invoice_items (54,873 rows): invoice_no FK, stock_code FK, quantity, unit_price, line_total\n  patient_customer_bridge (100 rows): links customer_id to MIMIC subject_id\nRelationships: invoice_items.invoice_no -> invoices | invoice_items.stock_code -> products | invoices.customer_id -> customers\nKey verified facts (from actual SQL on real database):\n  - 4,372 customers across 38 countries\n  - Confirmed order revenue (is_cancelled=0): £1,276,568.60\n  - Cancellation rate: 14.8% (3,836 of 25,900 invoices)\n  - Average confirmed order value: £100.52 per invoice\n  - Top 5 countries by revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194\n  - Date range: 2010-12-01 to 2011-12-09\n\nDATABASE 2 - MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):\nTables:\n  patients (100 rows): subject_id PK, gender M/F, anchor_age, anchor_year_group, dod\n  admissions (275 rows): hadm_id PK, subject_id FK, admission_type, insurance, hospital_expire_flag (1=died)\n  diagnoses_icd (4,506 rows): seq_num=1 is primary diagnosis, icd_version is 9 or 10\n  labevents (107,727 rows): itemid FK, value, valuenum, flag='abnormal' means outside reference range\n  prescriptions (18,087 rows): drug, dose, route, starttime, stoptime\n  icustays (140 rows): stay_id PK, first_careunit, los (length of stay in days)\n  chartevents (668,862 rows): ICU vital signs and physiological measurements\n  + d_labitems, d_icd_diagnoses, d_items reference dictionaries + 18 more tables\nKey verified facts (from actual SQL on real database):\n  - 100 patients: 43 female, 57 male. Mean anchor age: 61.8 years\n  - 31 patients have a recorded date of death (dod is not empty)\n  - In-hospital mortality: 15 deaths / 275 admissions = 5.5%\n  - Average ICU length of stay: 3.68 days (140 stays total)\n  - 107,727 lab results; 37.4% flagged as abnormal (40,275 results)\n  - 18,087 prescriptions | 668,862 chartevents\n  - Admission types: EW EMER, URGENT, ELECTIVE, DIRECT EMER, OBSERVATION ADMIT\n\nCROSS-DATABASE: patient_customer_bridge (100 rows) maps retail customer_id to MIMIC subject_id.\n\nAnswer all questions using these verified facts only."
      },
      {
        "role": "user",
        "content": "How many retail customers are linked to MIMIC patients?"
      },
      {
        "role": "assistant",
        "content": "100 retail customers linked to MIMIC patient records via patient_customer_bridge."
      }
    ]
  },
  {
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant trained on two real relational databases.\n\nDATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):\nTables:\n  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)\n  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)\n  invoices (25,900 rows): invoice_no PK, customer_id FK, invoice_date, is_cancelled (1=return/refund)\n  invoice_items (54,873 rows): invoice_no FK, stock_code FK, quantity, unit_price, line_total\n  patient_customer_bridge (100 rows): links customer_id to MIMIC subject_id\nRelationships: invoice_items.invoice_no -> invoices | invoice_items.stock_code -> products | invoices.customer_id -> customers\nKey verified facts (from actual SQL on real database):\n  - 4,372 customers across 38 countries\n  - Confirmed order revenue (is_cancelled=0): £1,276,568.60\n  - Cancellation rate: 14.8% (3,836 of 25,900 invoices)\n  - Average confirmed order value: £100.52 per invoice\n  - Top 5 countries by revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194\n  - Date range: 2010-12-01 to 2011-12-09\n\nDATABASE 2 - MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):\nTables:\n  patients (100 rows): subject_id PK, gender M/F, anchor_age, anchor_year_group, dod\n  admissions (275 rows): hadm_id PK, subject_id FK, admission_type, insurance, hospital_expire_flag (1=died)\n  diagnoses_icd (4,506 rows): seq_num=1 is primary diagnosis, icd_version is 9 or 10\n  labevents (107,727 rows): itemid FK, value, valuenum, flag='abnormal' means outside reference range\n  prescriptions (18,087 rows): drug, dose, route, starttime, stoptime\n  icustays (140 rows): stay_id PK, first_careunit, los (length of stay in days)\n  chartevents (668,862 rows): ICU vital signs and physiological measurements\n  + d_labitems, d_icd_diagnoses, d_items reference dictionaries + 18 more tables\nKey verified facts (from actual SQL on real database):\n  - 100 patients: 43 female, 57 male. Mean anchor age: 61.8 years\n  - 31 patients have a recorded date of death (dod is not empty)\n  - In-hospital mortality: 15 deaths / 275 admissions = 5.5%\n  - Average ICU length of stay: 3.68 days (140 stays total)\n  - 107,727 lab results; 37.4% flagged as abnormal (40,275 results)\n  - 18,087 prescriptions | 668,862 chartevents\n  - Admission types: EW EMER, URGENT, ELECTIVE, DIRECT EMER, OBSERVATION ADMIT\n\nCROSS-DATABASE: patient_customer_bridge (100 rows) maps retail customer_id to MIMIC subject_id.\n\nAnswer all questions using these verified facts only."
      },
      {
        "role": "user",
        "content": "What tables are in the MIMIC-IV database?"
      },
      {
        "role": "assistant",
        "content": "MIMIC-IV has 28+ tables. Core tables: patients (100 rows), admissions (275 rows), diagnoses_icd (4,506 rows), labevents (107,727 rows), prescriptions (18,087 rows), icustays (140 rows), chartevents (668,862 rows), plus reference dictionaries d_labitems, d_icd_diagnoses, d_items, and more including procedures, transfers, emar, pharmacy."
      }
    ]
  },
  {
    "messages": [
      {
        "role": "system",
        "content": "You are an AI assistant trained on two real relational databases.\n\nDATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):\nTables:\n  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)\n  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)\n  invoices (25,900 rows): invoice_no PK, customer_id FK, invoice_date, is_cancelled (1=return/refund)\n  invoice_items (54,873 rows): invoice_no FK, stock_code FK, quantity, unit_price, line_total\n  patient_customer_bridge (100 rows): links customer_id to MIMIC subject_id\nRelationships: invoice_items.invoice_no -> invoices | invoice_items.stock_code -> products | invoices.customer_id -> customers\nKey verified facts (from actual SQL on real database):\n  - 4,372 customers across 38 countries\n  - Confirmed order revenue (is_cancelled=0): £1,276,568.60\n  - Cancellation rate: 14.8% (3,836 of 25,900 invoices)\n  - Average confirmed order value: £100.52 per invoice\n  - Top 5 countries by revenue: United Kingdom £685,023 | Germany £38,665 | France £35,492 | EIRE £22,848 | Spain £18,194\n  - Date range: 2010-12-01 to 2011-12-09\n\nDATABASE 2 - MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):\nTables:\n  patients (100 rows): subject_id PK, gender M/F, anchor_age, anchor_year_group, dod\n  admissions (275 rows): hadm_id PK, subject_id FK, admission_type, insurance, hospital_expire_flag (1=died)\n  diagnoses_icd (4,506 rows): seq_num=1 is primary diagnosis, icd_version is 9 or 10\n  labevents (107,727 rows): itemid FK, value, valuenum, flag='abnormal' means outside reference range\n  prescriptions (18,087 rows): drug, dose, route, starttime, stoptime\n  icustays (140 rows): stay_id PK, first_careunit, los (length of stay in days)\n  chartevents (668,862 rows): ICU vital signs and physiological measurements\n  + d_labitems, d_icd_diagnoses, d_items reference dictionaries + 18 more tables\nKey verified facts (from actual SQL on real database):\n  - 100 patients: 43 female, 57 male. Mean anchor age: 61.8 years\n  - 31 patients have a recorded date of death (dod is not empty)\n  - In-hospital mortality: 15 deaths / 275 admissions = 5.5%\n  - Average ICU length of stay: 3.68 days (140 stays total)\n  - 107,727 lab results; 37.4% flagged as abnormal (40,275 results)\n  - 18,087 prescriptions | 668,862 chartevents\n  - Admission types: EW EMER, URGENT, ELECTIVE, DIRECT EMER, OBSERVATION ADMIT\n\nCROSS-DATABASE: patient_customer_bridge (100 rows) maps retail customer_id to MIMIC subject_id.\n\nAnswer all questions using these verified facts only."
      },
      {
        "role": "user",
        "content": "What is the relationship between admissions, icustays, and chartevents?"
      },
      {
        "role": "assistant",
        "content": "Hierarchy: patients (subject_id) -> admissions (hadm_id) -> icustays (stay_id) -> chartevents. One patient -> multiple admissions -> multiple ICU stays -> thousands of chart measurements each."
      }
    ]
  }
]

random.shuffle(EXAMPLES)
split = int(len(EXAMPLES) * 0.9)
train_data = EXAMPLES[:split]
val_data   = EXAMPLES[split:]
print(f'Total: {len(EXAMPLES)} examples | Train: {len(train_data)} | Val: {len(val_data)}')
print(f'\nSample Q: {EXAMPLES[0]["messages"][1]["content"]}')
print(f'Sample A: {EXAMPLES[0]["messages"][2]["content"][:120]}...')


Total: 33 examples | Train: 29 | Val: 4

Sample Q: What does the 14.8% cancellation rate tell us about the retail business?
Sample A: 3,836 of 25,900 invoices are returns/cancellations. Revenue analysis must filter is_cancelled=0. Confirmed revenue is £1...


## Step 4: Load Model with 4-bit Quantization

We support `Qwen/Qwen2.5-Coder-7B-Instruct` (specialized for SQL/database reasoning) or `microsoft/Phi-3.5-mini-instruct` (3.8B lightweight). Both fit comfortably on a free Google Colab T4 GPU in 4-bit (~5 GB VRAM).

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# Option A (Recommended for SQL/DB reasoning): Qwen2.5-Coder-7B
MODEL_ID = 'Qwen/Qwen2.5-Coder-7B-Instruct'
# Option B (Lightweight 3.8B alternative):
# MODEL_ID = 'microsoft/Phi-3.5-mini-instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token if tokenizer.eos_token is not None else tokenizer.pad_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model.config.use_cache = False
total = sum(p.numel() for p in model.parameters())
print(f'Loaded {MODEL_ID}: {total/1e9:.2f}B parameters')

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loaded Qwen/Qwen2.5-Coder-7B-Instruct: 4.35B parameters


## Step 5: Baseline Test (BEFORE Training)

In [5]:
def ask(question, use_system=False, max_new_tokens=150):
    sys_msg = 'You are a helpful AI assistant.'
    if use_system:
        sys_msg = EXAMPLES[0]['messages'][0]['content']  # DB-specific system prompt
    messages = [{'role': 'system', 'content': sys_msg}, {'role': 'user', 'content': question}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            use_cache=False,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

TEST_QUESTIONS = [
    'How many patients are in the MIMIC-IV demo database?',           # answer: 100
    'What is the in-hospital mortality rate in MIMIC?',               # answer: 5.5%
    'How many unique customers are in the Online Retail database?',   # answer: 4,372
    'What does hospital_expire_flag mean?',
    'Which country generates the most retail revenue?',               # answer: United Kingdom
]

print('=== BASE MODEL (before fine-tuning) ===')
baseline = {}
for q in TEST_QUESTIONS:
    a = ask(q)
    baseline[q] = a
    print(f'Q: {q}')
    print(f'A: {a[:200]}')
    print()

=== BASE MODEL (before fine-tuning) ===
Q: How many patients are in the MIMIC-IV demo database?
A: The MIMIC-IV demo database contains 10,000 patients.

Q: What is the in-hospital mortality rate in MIMIC?
A: The in-hospital mortality rate in MIMIC (Medical Information Mart for Intensive Care) varies depending on the specific version of the dataset and the criteria used to define mortality. However, as of 

Q: How many unique customers are in the Online Retail database?
A: To determine the number of unique customers in the Online Retail database, you would need to analyze the customer data stored in the database. This typically involves querying the database to count di

Q: What does hospital_expire_flag mean?
A: The "hospital_expire_flag" is a variable used in medical records systems to indicate whether a patient has died while in the hospital or if they have been discharged alive. This flag helps healthcare 

Q: Which country generates the most retail revenue?
A: As of my last update 

## Step 6: Apply QLoRA Adapters

In [6]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# Prepare model for 4-bit quantization training
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,              # LoRA rank
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
)
model = get_peft_model(model, lora_config)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.3f}%)')

Trainable: 10,092,544 / 4,363,064,832 (0.231%)


## Step 7: Prepare Dataset

In [7]:
def format_example(ex):
    try:
        return tokenizer.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False)
    except Exception:
        msgs = ex['messages']
        s = next(m['content'] for m in msgs if m['role']=='system')
        u = next(m['content'] for m in msgs if m['role']=='user')
        a = next(m['content'] for m in msgs if m['role']=='assistant')
        return f'<|im_start|>system\n{s}<|im_end|>\n<|im_start|>user\n{u}<|im_end|>\n<|im_start|>assistant\n{a}<|im_end|>'

# Pure Python dataset structure (No pyarrow/C-extension dependencies)
train_dataset = [{'text': format_example(e)} for e in train_data]
val_dataset   = [{'text': format_example(e)} for e in val_data]

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)}')
print(f'Sample (first 400 chars):\n{train_dataset[0]["text"][:400]}')


Train: 29 | Val: 4
Sample (first 400 chars):
<|im_start|>system
You are an AI assistant trained on two real relational databases.

DATABASE 1 - Online Retail (UK e-commerce, December 2010 - December 2011):
Tables:
  customers (4,372 rows): customer_id PK, country, segment (Domestic_UK/EU_Wholesale/International)
  products (1,124 rows): stock_code PK, description, category, unit_price (GBP)
  invoices (25,900 rows): invoice_no PK, customer_i


## Step 8: Fine-Tune

In [8]:
import torch
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

# Tokenize texts directly with PyTorch (max_length=512 preserves memory)
train_enc = tokenizer([d['text'] for d in train_dataset], truncation=True, max_length=512)
val_enc   = tokenizer([d['text'] for d in val_dataset], truncation=True, max_length=512)

class CausalDataset(torch.utils.data.Dataset):
    def __init__(self, enc):
        self.input_ids = enc['input_ids']
        self.attention_mask = enc['attention_mask']
    def __len__(self):
        return len(self.input_ids)
    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.input_ids[idx]),
            'attention_mask': torch.tensor(self.attention_mask[idx]),
            'labels': torch.tensor(self.input_ids[idx])
        }

train_ds = CausalDataset(train_enc)
val_ds   = CausalDataset(val_enc)

training_args = TrainingArguments(
    output_dir='/content/db_slm_adapter',
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=5,
    fp16=True,
    logging_steps=5,
    save_strategy='epoch',
    report_to='none',
    gradient_checkpointing=True,        # Cuts activation memory by ~70%
    optim='paged_adamw_8bit',          # 8-bit optimizer saves VRAM
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

print(f'Training {len(train_ds)} examples for {training_args.num_train_epochs} epochs...')
result = trainer.train()
print(f'\nDone! Loss: {result.training_loss:.4f} | Steps: {result.global_step} | Time: {result.metrics.get("train_runtime",0):.0f}s')


Training 29 examples for 5 epochs...


Step,Training Loss
5,1.925856
10,1.568785
15,1.095020
20,0.669437
25,0.363927
30,0.231220
35,0.169098
40,0.148592



Done! Loss: 0.7715 | Steps: 40 | Time: 290s


## Step 9: Test Fine-Tuned Model (AFTER Training)

In [9]:
def ask_ft(question, max_new_tokens=200):
    # Use the DB-specific system prompt after fine-tuning
    messages = [
        {'role': 'system', 'content': EXAMPLES[0]['messages'][0]['content']},
        {'role': 'user', 'content': question}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            use_cache=False,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

print('=== FINE-TUNED MODEL (after training) ===')
finetuned = {}
for q in TEST_QUESTIONS:
    a = ask_ft(q)
    finetuned[q] = a
    print(f'Q: {q}')
    print(f'A: {a[:200]}')
    print()

=== FINE-TUNED MODEL (after training) ===
Q: How many patients are in the MIMIC-IV demo database?
A: There are 100 patients in the MIMIC-IV clinical demo database.

Q: What is the in-hospital mortality rate in MIMIC?
A: The in-hospital mortality rate in MIMIC is 5.5%. This can be calculated by dividing the number of deaths (31 out of 100 patients) by the total number of patients.

Q: How many unique customers are in the Online Retail database?
A: There are 4,372 unique customers in the Online Retail database.

Q: What does hospital_expire_flag mean?
A: In the MIMIC-IV Clinical Demo database, the `hospital_expire_flag` column in the `admissions` table indicates whether a patient died during their hospital stay:

- A value of **1** means the patient e

Q: Which country generates the most retail revenue?
A: The United Kingdom generates the most retail revenue with £685,023 confirmed order value.



## Step 10: Before vs After Evaluation

In [10]:
EVAL = [
    {'q': 'How many patients in MIMIC?',                 'must': ['100'],             'num': 100},
    {'q': 'What is the in-hospital mortality rate?',     'must': ['5.5', '15'],       'num': 5.5},
    {'q': 'How many hospital admissions in MIMIC?',      'must': ['275'],             'num': 275},
    {'q': 'How many ICU stays in MIMIC?',                'must': ['140'],             'num': 140},
    {'q': 'Average ICU length of stay?',                 'must': ['3.68', '3.7'],     'num': 3.68},
    {'q': 'How many unique customers in retail?',        'must': ['4,372', '4372'],   'num': 4372},
    {'q': 'How many invoices in retail?',                'must': ['25,900', '25900'], 'num': 25900},
    {'q': 'What is the confirmed revenue from retail?',  'must': ['1,276', '1276'],   'num': 1276568},
    {'q': 'What percent of lab results are abnormal?',   'must': ['37.4', '37%'],     'num': 37.4},
    {'q': 'How many patients have recorded death date?', 'must': ['31'],              'num': 31},
    {'q': 'What tables are in retail database?',         'must': ['invoice_items', 'customers', 'products'], 'num': None},
    {'q': 'What does hospital_expire_flag mean?',        'must': ['death', 'died', 'mortality'], 'num': None},
    {'q': 'Which country earns most retail revenue?',    'must': ['United Kingdom', 'UK'], 'num': None},
]

def score(ans, case):
    return sum(1 for k in case['must'] if k.lower() in ans.lower()) / len(case['must'])

print(f'{"Question":<52} {"Before":>8} {"After":>7}')
print('-'*70)
b_scores, f_scores = [], []
for c in EVAL:
    b = ask(c['q'])
    f = ask_ft(c['q'])
    bs = score(b, c)
    fs = score(f, c)
    b_scores.append(bs)
    f_scores.append(fs)
    bi = '✅' if bs>=0.7 else ('⚠️' if bs>=0.4 else '❌')
    fi = '✅' if fs>=0.7 else ('⚠️' if fs>=0.4 else '❌')
    print(f"{c['q'][:50]:<52} {bi}{bs:.0%}  {fi}{fs:.0%}")

b_avg = sum(b_scores)/len(b_scores)
f_avg = sum(f_scores)/len(f_scores)
b_pass = sum(1 for s in b_scores if s>=0.7)
f_pass = sum(1 for s in f_scores if s>=0.7)
print('-'*70)
print(f'{'TOTAL':<52} {b_pass}/{len(EVAL)} {b_avg:.0%}  {f_pass}/{len(EVAL)} {f_avg:.0%}')
print(f'\nImprovement: {b_avg:.0%} → {f_avg:.0%} (+{(f_avg-b_avg):.0%})')

Question                                               Before   After
----------------------------------------------------------------------
How many patients in MIMIC?                          ❌0%  ✅100%
What is the in-hospital mortality rate?              ❌0%  ⚠️50%
How many hospital admissions in MIMIC?               ❌0%  ✅100%
How many ICU stays in MIMIC?                         ❌0%  ✅100%
Average ICU length of stay?                          ❌0%  ⚠️50%
How many unique customers in retail?                 ❌0%  ⚠️50%
How many invoices in retail?                         ❌0%  ⚠️50%
What is the confirmed revenue from retail?           ❌0%  ⚠️50%
What percent of lab results are abnormal?            ❌0%  ⚠️50%
How many patients have recorded death date?          ❌0%  ✅100%
What tables are in retail database?                  ⚠️67%  ✅100%
What does hospital_expire_flag mean?                 ❌33%  ⚠️67%
Which country earns most retail revenue?             ❌0%  ⚠️50%
------------------------

## Step 11: Save Adapter & Download

In [11]:
import zipfile, json
from pathlib import Path

SAVE_DIR = '/content/db_slm_adapter'
trainer.model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

summary = {
    'base_model': MODEL_ID, 'epochs': 5, 'train_examples': len(train_data),
    'lora_rank': 16, 'train_loss': result.training_loss,
    'baseline_score': f'{b_avg:.1%}', 'finetuned_score': f'{f_avg:.1%}',
    'improvement': f'+{(f_avg-b_avg):.1%}',
    'verified_facts': {
        'retail_confirmed_revenue': '£1,276,568.60',
        'mimic_mortality_rate': '5.5% (15/275)',
        'mimic_patients': '100 (43F, 57M)',
        'avg_icu_los': '3.68 days',
    }
}
with open(f'{SAVE_DIR}/training_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

zip_path = '/content/db_slm_adapter.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for fp in Path(SAVE_DIR).rglob('*'):
        zf.write(fp, fp.relative_to('/content'))

print(f'Saved: {SAVE_DIR}')
print(json.dumps(summary, indent=2))
print('\nDownload: Files panel (left) → db_slm_adapter.zip')

Saved: /content/db_slm_adapter
{
  "base_model": "Qwen/Qwen2.5-Coder-7B-Instruct",
  "epochs": 5,
  "train_examples": 29,
  "lora_rank": 16,
  "train_loss": 0.7714917972683907,
  "baseline_score": "7.7%",
  "finetuned_score": "70.5%",
  "improvement": "+62.8%",
  "verified_facts": {
    "retail_confirmed_revenue": "\u00a31,276,568.60",
    "mimic_mortality_rate": "5.5% (15/275)",
    "mimic_patients": "100 (43F, 57M)",
    "avg_icu_los": "3.68 days"
  }
}

Download: Files panel (left) → db_slm_adapter.zip


## Step 12: Chat with Your Trained Model

In [13]:
# Change this question and run the cell
question = 'Can you explain me the schema of online retail database?'

print(f'Q: {question}')
print(f'A: {ask_ft(question)}')

Q: Can you explain me the schema of online retail database?
A: Certainly! The Online Retail database is a simulated e-commerce dataset from a UK retailer between December 2010 and December 2011. Here's a breakdown of the key tables and their relationships:

1. **Customers Table**:
   - **customer_id**: Primary Key, unique identifier for each customer.
   - **country**: Indicates the country where the customer is located.
   - **segment**: Categorizes the customer into one of three segments: Domestic_UK, EU_Wholesale, or International.

2. **Products Table**:
   - **stock_code**: Primary Key, unique identifier for each product.
   - **description**: Descriptive name of the product.
   - **category**: Broad category of the product.
   - **unit_price**: Price per unit of the product in GBP.

3. **Invoices Table**:
   - **invoice_no**: Primary Key, unique identifier for each invoice.
   - **customer_id**: Foreign Key linking to
